# Grammar Scoring Engine from Spoken Audio

An end-to-end multimodal solution for predicting a continuous grammar score from a 45–60 second speech sample. The final model combines frozen WavLM audio representations with three transcript views: TF‑IDF, MPNet, and DeBERTa.

**Last full Kaggle run:** NVIDIA Tesla T4, Python 3.12, 5-fold out-of-fold (OOF) evaluation on the 732 rubric-valid training examples.

| Final OOF metric | Score |
|---|---:|
| RMSE | **0.532446** |
| Pearson correlation | **0.851207** |

The notebook is deliberately linear and can be run with **Save & Run All** after enabling a GPU and attaching the competition dataset. Expensive feature stages write resumable caches to `/kaggle/working`.


## 1. Environment and reproducibility

I install the few packages missing from the Kaggle image in one place. A GPU is required: Whisper transcription, DeBERTa fine-tuning, and WavLM embedding extraction are the expensive stages.


In [1]:
%pip install -q faster-whisper sentence-transformers sentencepiece soundfile

In [2]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

SEED = 42
N_SPLITS = 5

DATA_DIR = Path("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final")
WORK_DIR = Path("/kaggle/working")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"


def seed_everything(seed: int = SEED) -> None:
    # Seed every stochastic library used below.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

required_paths = [
    DATA_DIR / "train.csv",
    DATA_DIR / "test.csv",
    DATA_DIR / "sample_submission.csv",
    TRAIN_DIR,
    TEST_DIR,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Attach the competition dataset before running. Missing: "
        + ", ".join(missing_paths)
    )

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a Kaggle GPU accelerator.")

print(f"Python work directory: {WORK_DIR}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data audit and modeling decision

The supplied files contain 769 training clips and 216 test clips. Two anomalies materially affect the pipeline:

1. The 37 labels equal to `0` sit outside the stated 1–5 grammar rubric, share a separate filename range, and have much lower ASR confidence. I treat them as an audio-quality/source regime and train the grammar regressor on the 732 positive-label rows. This avoids learning a filename/source shortcut, but it means the final model intentionally predicts only the rubric range `[1, 5]`.
2. `sample_submission.csv` contains 204 rows and does not match the test audio directory. `test.csv` matches all 216 test files exactly, so it is the validated source of submission order.

Train and test reuse many basenames for different audio, so every cache is keyed by `(split, filename)` rather than filename alone.


In [3]:
import soundfile as sf

train_csv = pd.read_csv(DATA_DIR / "train.csv")
test_csv = pd.read_csv(DATA_DIR / "test.csv")
sample_csv = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert list(train_csv.columns) == ["filename", "label"]
assert "filename" in test_csv.columns
assert train_csv["filename"].is_unique
assert test_csv["filename"].is_unique
assert train_csv["label"].notna().all()

train_files = {path.name for path in TRAIN_DIR.glob("*.wav")}
test_files = {path.name for path in TEST_DIR.glob("*.wav")}
assert set(train_csv["filename"]) == train_files
assert set(test_csv["filename"]) == test_files

audio_info = []
for split, directory in (("train", TRAIN_DIR), ("test", TEST_DIR)):
    for path in directory.glob("*.wav"):
        info = sf.info(path)
        audio_info.append(
            {
                "split": split,
                "filename": path.name,
                "duration_sec": info.duration,
                "sample_rate": info.samplerate,
                "channels": info.channels,
            }
        )
audio_info = pd.DataFrame(audio_info)

target_counts = train_csv["label"].value_counts().sort_index()
zero_count = int((train_csv["label"] == 0).sum())

print(f"Train/test rows: {len(train_csv)}/{len(test_csv)}")
print(f"Sample-submission rows: {len(sample_csv)}")
print(f"Zero-label training rows: {zero_count}")
print(
    f"Audio duration: {audio_info['duration_sec'].min():.2f}–{audio_info['duration_sec'].max():.2f}s"
)
print("Target distribution:\n", target_counts.to_string())

if len(sample_csv) != len(test_csv) or set(sample_csv["filename"]) != test_files:
    print(
        "WARNING: sample_submission.csv is inconsistent with test.csv/test audio. "
        "The submission is therefore built in validated test.csv order."
    )

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(target_counts.index.astype(str), target_counts.values, color="#4472C4")
axes[0].set(
    title="Training target distribution", xlabel="Grammar score", ylabel="Count"
)
axes[1].hist(
    [
        audio_info.loc[audio_info["split"] == "train", "duration_sec"],
        audio_info.loc[audio_info["split"] == "test", "duration_sec"],
    ],
    bins=20,
    label=["train", "test"],
    alpha=0.75,
)
axes[1].set(title="Audio duration", xlabel="Seconds", ylabel="Count")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Experiment path

I started with simple acoustic statistics to establish a baseline, then added richer representations only when cross-validation improved.

| Experiment | OOF RMSE | Pearson | Decision |
|---|---:|---:|---|
| Mean baseline (positive labels) | 1.01378 | — | Reference only |
| Handcrafted audio + ExtraTrees | 0.84826 | 0.55186 | Useful, but source-sensitive |
| TF‑IDF + ASR/linguistic features | 0.68057 | 0.74179 | Keep |
| MPNet + numeric features | 0.65845 | 0.76051 | Keep |
| 60% MPNet / 40% TF‑IDF | 0.64538 | 0.77175 | Keep |
| + DeBERTa (cross-fitted calibration) | 0.62408 | 0.78805 | Keep for complementary errors |
| + WavLM, nested blend selection | **0.53245** | **0.85121** | Final |

Model and blend choices use OOF predictions. Affine calibration is cross-fitted when it is evaluated, so validation labels are never used to fit their own calibration parameters.


## 4. Whisper transcription

Grammar is primarily expressed in words and syntax, so I use `faster-whisper` (`medium.en`) to produce transcripts plus ASR-confidence and timing features. The cache is checkpointed after every 25 files, making this longest stage safe to resume.


In [4]:
from pathlib import Path
import json
import time
import traceback

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from faster_whisper import WhisperModel

# CONFIG

DATA_DIR = Path("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

OUTPUT_PATH = Path("/kaggle/working/asr_transcripts_medium_en.csv")

MODEL_NAME = "medium.en"


# 1. LOAD WHISPER

print("=" * 80)
print("LOADING WHISPER")
print("=" * 80)

print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))

model = WhisperModel(
    MODEL_NAME,
    device="cuda",
    compute_type="float16",
    download_root="/kaggle/working/whisper_models",
)

print("Model loaded successfully.")


# 2. BUILD EXACT FILE LIST
#
# IMPORTANT:
# train/audio_10.wav and test/audio_10.wav can be different.
# Therefore split + filename is our unique key.

jobs = []

for _, row in train.iterrows():
    jobs.append(
        {
            "split": "train",
            "filename": row["filename"],
            "path": TRAIN_DIR / row["filename"],
        }
    )

for _, row in test.iterrows():
    jobs.append(
        {
            "split": "test",
            "filename": row["filename"],
            "path": TEST_DIR / row["filename"],
        }
    )


print("\nTotal files:", len(jobs))
print("Train:", len(train))
print("Test :", len(test))


# 3. LOAD EXISTING CHECKPOINT IF PRESENT
#
# Makes the process resumable if the notebook stops.

if OUTPUT_PATH.exists():

    existing = pd.read_csv(OUTPUT_PATH)

    done_keys = set(zip(existing["split"], existing["filename"]))

    records = existing.to_dict("records")

    print(f"\nResuming from checkpoint: " f"{len(records)} files already completed.")

else:

    done_keys = set()
    records = []

    print("\nStarting new transcription cache.")


# 4. TRANSCRIPTION FUNCTION


def transcribe_file(path):

    segments_gen, info = model.transcribe(
        str(path),
        language="en",
        # Good accuracy/speed compromise
        beam_size=5,
        # Helps prevent silence/noise hallucinations
        vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 500},
        # Useful diagnostic information
        word_timestamps=True,
    )

    segments = list(segments_gen)

    texts = []
    avg_logprobs = []
    no_speech_probs = []

    all_words = []

    for seg in segments:

        text = seg.text.strip()

        if text:
            texts.append(text)

        if hasattr(seg, "avg_logprob"):
            avg_logprobs.append(float(seg.avg_logprob))

        if hasattr(seg, "no_speech_prob"):
            no_speech_probs.append(float(seg.no_speech_prob))

        if seg.words is not None:

            for word in seg.words:

                all_words.append(
                    {
                        "word": word.word.strip(),
                        "start": (
                            float(word.start) if word.start is not None else None
                        ),
                        "end": (float(word.end) if word.end is not None else None),
                        "probability": (
                            float(word.probability)
                            if word.probability is not None
                            else None
                        ),
                    }
                )

    transcript = " ".join(texts).strip()

    # Word timing statistics

    valid_words = [
        w for w in all_words if (w["start"] is not None and w["end"] is not None)
    ]

    if valid_words:

        first_word_start = valid_words[0]["start"]
        last_word_end = valid_words[-1]["end"]

        speech_span = last_word_end - first_word_start

        mean_word_prob = np.mean(
            [w["probability"] for w in valid_words if w["probability"] is not None]
        )

    else:

        first_word_start = np.nan
        last_word_end = np.nan
        speech_span = 0.0
        mean_word_prob = np.nan

    return {
        "transcript": transcript,
        "detected_language": info.language,
        "language_probability": float(info.language_probability),
        "audio_duration": float(info.duration),
        "segment_count": len(segments),
        "word_count": len(valid_words),
        "char_count": len(transcript),
        "avg_logprob": np.mean(avg_logprobs) if avg_logprobs else np.nan,
        "avg_no_speech_prob": np.mean(no_speech_probs) if no_speech_probs else np.nan,
        "mean_word_probability": mean_word_prob,
        "first_word_start": first_word_start,
        "last_word_end": last_word_end,
        "speech_span": speech_span,
        # Keep timings so we can later derive
        # speaking-rate / pause features without
        # retranscribing.
        "words_json": json.dumps(all_words, ensure_ascii=False),
    }


# 5. RUN ASR

remaining = [job for job in jobs if (job["split"], job["filename"]) not in done_keys]


print("\n" + "=" * 80)
print("RUNNING ASR")
print("=" * 80)

print("Remaining files:", len(remaining))


for idx, job in enumerate(tqdm(remaining), start=1):

    row = {"split": job["split"], "filename": job["filename"]}

    try:

        result = transcribe_file(job["path"])

        row.update(result)

        row["error"] = ""

    except Exception as e:

        print("\nERROR:", job["split"], job["filename"], repr(e))

        row.update(
            {
                "transcript": "",
                "detected_language": "",
                "language_probability": np.nan,
                "audio_duration": np.nan,
                "segment_count": 0,
                "word_count": 0,
                "char_count": 0,
                "avg_logprob": np.nan,
                "avg_no_speech_prob": np.nan,
                "mean_word_probability": np.nan,
                "first_word_start": np.nan,
                "last_word_end": np.nan,
                "speech_span": np.nan,
                "words_json": "[]",
                "error": repr(e),
            }
        )

    records.append(row)

    # checkpoint frequently
    if idx % 20 == 0 or idx == len(remaining):

        pd.DataFrame(records).to_csv(OUTPUT_PATH, index=False)


print("\nSaved:")
print(OUTPUT_PATH)


# 6. LOAD FINAL CACHE

asr = pd.read_csv(OUTPUT_PATH)

print("\n" + "=" * 80)
print("ASR CACHE SUMMARY")
print("=" * 80)

print("Shape:", asr.shape)

print("\nSplit counts:")
print(asr["split"].value_counts())

print("\nTranscription errors:", (asr["error"].fillna("") != "").sum())

print("Empty transcripts:", (asr["transcript"].fillna("").str.strip() == "").sum())


# 7. BASIC TRANSCRIPT STATISTICS

print("\n" + "=" * 80)
print("TRANSCRIPT STATISTICS")
print("=" * 80)

for split in ["train", "test"]:

    temp = asr[asr["split"] == split]

    print(f"\n{split.upper()}")

    print(
        temp[
            [
                "word_count",
                "char_count",
                "language_probability",
                "mean_word_probability",
            ]
        ]
        .describe()
        .round(4)
    )


# 8. JOIN TRAIN LABELS

train_asr = train.merge(
    asr[asr["split"] == "train"], on="filename", how="left", validate="one_to_one"
)


# 9. ASR QUALITY BY GRAMMAR SCORE

print("\n" + "=" * 80)
print("ASR SUMMARY BY LABEL")
print("=" * 80)

summary = train_asr.groupby("label").agg(
    n=("label", "size"),
    words=("word_count", "mean"),
    lang_prob=("language_probability", "mean"),
    word_prob=("mean_word_probability", "mean"),
    avg_logprob=("avg_logprob", "mean"),
)

print(summary.round(4).to_string())


# 10. ZERO-SCORE INSPECTION

print("\n" + "=" * 80)
print("ZERO-SCORE TRANSCRIPT EXAMPLES")
print("=" * 80)

zero_examples = train_asr[train_asr["label"] == 0].head(8)

for _, row in zero_examples.iterrows():

    print("\n---")
    print("FILE:", row["filename"])
    print("WORDS:", row["word_count"])
    print(
        "WORD PROB:",
        (
            round(row["mean_word_probability"], 4)
            if pd.notna(row["mean_word_probability"])
            else None
        ),
    )

    print("TEXT:", str(row["transcript"])[:700])


# 11. NORMAL SCORE EXAMPLES

print("\n" + "=" * 80)
print("TRANSCRIPT EXAMPLES BY SCORE")
print("=" * 80)

for score in [2.0, 3.0, 4.0, 5.0]:

    subset = train_asr[train_asr["label"] == score]

    if len(subset) == 0:
        continue

    example = subset.sample(1, random_state=42).iloc[0]

    print("\n" + "-" * 80)
    print(
        "SCORE:",
        score,
        "| FILE:",
        example["filename"],
        "| WORDS:",
        example["word_count"],
    )

    print(str(example["transcript"])[:1000])


print("\n" + "=" * 80)
print("TRANSCRIPTION STAGE COMPLETE")
print("=" * 80)

ASR cache created for 769 train and 216 test clips.


## 5. Sparse text baseline

Word bigrams capture local syntax, character n-grams capture morphology and ASR-rendered contractions, and interpretable numeric features describe fluency, lexical diversity, fillers, conjunctions, and speaking rate. All vectorizers and scalers are fit inside each fold.


In [5]:
# STEP 5 — FIRST SERIOUS TEXT MODEL
#
# Model:
#   transcript
#      -> word TF-IDF
#      -> character TF-IDF
#   ASR / linguistic statistics
#      -> scaled numeric features
#   all features
#      -> Ridge regression
#
# Validation:
#   only label > 0 examples
#   same 5-fold stratification philosophy as before

from pathlib import Path
import re
import json
import warnings


import numpy as np
import pandas as pd

from scipy import sparse
from scipy.stats import pearsonr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

# CONFIG

SEED = 42
N_SPLITS = 5

DATA_DIR = Path("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final")

ASR_PATH = Path("/kaggle/working/asr_transcripts_medium_en.csv")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
asr = pd.read_csv(ASR_PATH)


# 1. JOIN ASR DATA

train_asr = train.merge(
    asr[asr["split"] == "train"], on="filename", how="left", validate="one_to_one"
)

test_asr = test.merge(
    asr[asr["split"] == "test"], on="filename", how="left", validate="one_to_one"
)

train_asr["transcript"] = train_asr["transcript"].fillna("").astype(str)

test_asr["transcript"] = test_asr["transcript"].fillna("").astype(str)


print("=" * 80)
print("JOIN CHECK")
print("=" * 80)

print("Train:", train_asr.shape)
print("Test :", test_asr.shape)

print("Missing train transcripts:", (train_asr["transcript"].str.strip() == "").sum())

print("Missing test transcripts :", (test_asr["transcript"].str.strip() == "").sum())


# 2. SIMPLE LINGUISTIC FEATURES
#
# These are deliberately interpretable.

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

FILLERS = {"um", "uh", "erm", "hmm", "ah", "huh"}

SUBORDINATORS = {
    "because",
    "although",
    "though",
    "while",
    "whereas",
    "unless",
    "since",
    "until",
    "before",
    "after",
    "when",
    "whenever",
    "where",
    "wherever",
    "if",
    "whether",
}

COORDINATORS = {"and", "but", "or", "so", "yet"}


def linguistic_features(text):

    text = str(text)

    words = [w.lower() for w in WORD_RE.findall(text)]

    n_words = len(words)

    if n_words == 0:
        return {
            "lex_unique_ratio": 0.0,
            "avg_word_length": 0.0,
            "long_word_ratio": 0.0,
            "filler_ratio": 0.0,
            "subordinator_ratio": 0.0,
            "coordinator_ratio": 0.0,
            "repeat_ratio": 0.0,
            "sentence_count": 0.0,
            "words_per_sentence": 0.0,
            "question_count": 0.0,
        }

    unique_ratio = len(set(words)) / n_words

    avg_word_length = np.mean([len(w) for w in words])

    long_word_ratio = np.mean([len(w) >= 7 for w in words])

    filler_ratio = sum(w in FILLERS for w in words) / n_words

    subordinator_ratio = sum(w in SUBORDINATORS for w in words) / n_words

    coordinator_ratio = sum(w in COORDINATORS for w in words) / n_words

    repeated = sum(words[i] == words[i - 1] for i in range(1, n_words))

    repeat_ratio = repeated / max(n_words - 1, 1)

    # Whisper punctuation gives us a rough sentence proxy.
    sentence_parts = [x.strip() for x in re.split(r"[.!?]+", text) if x.strip()]

    sentence_count = len(sentence_parts)

    words_per_sentence = n_words / sentence_count if sentence_count > 0 else n_words

    return {
        "lex_unique_ratio": unique_ratio,
        "avg_word_length": avg_word_length,
        "long_word_ratio": long_word_ratio,
        "filler_ratio": filler_ratio,
        "subordinator_ratio": subordinator_ratio,
        "coordinator_ratio": coordinator_ratio,
        "repeat_ratio": repeat_ratio,
        "sentence_count": sentence_count,
        "words_per_sentence": words_per_sentence,
        "question_count": text.count("?"),
    }


def build_numeric_features(df):

    ling = pd.DataFrame(
        [linguistic_features(t) for t in df["transcript"]], index=df.index
    )

    out = pd.DataFrame(index=df.index)

    raw_cols = [
        "word_count",
        "char_count",
        "segment_count",
        "audio_duration",
        "avg_logprob",
        "avg_no_speech_prob",
        "mean_word_probability",
        "first_word_start",
        "last_word_end",
        "speech_span",
    ]

    for col in raw_cols:
        out[col] = pd.to_numeric(df[col], errors="coerce")

    out["words_per_audio_sec"] = out["word_count"] / out["audio_duration"].clip(lower=1)

    out["words_per_speech_sec"] = out["word_count"] / out["speech_span"].clip(lower=1)

    out["chars_per_word"] = out["char_count"] / out["word_count"].clip(lower=1)

    out["speech_span_ratio"] = out["speech_span"] / out["audio_duration"].clip(lower=1)

    out = pd.concat([out, ling], axis=1)

    # Replace infinities / NaNs with column medians.
    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.columns:

        med = out[col].median()

        if pd.isna(med):
            med = 0.0

        out[col] = out[col].fillna(med)

    return out.astype(float)


train_num = build_numeric_features(train_asr)

test_num = build_numeric_features(test_asr)


print("\nNumeric feature count:", train_num.shape[1])

print("\nNumeric feature names:")
print(train_num.columns.tolist())


# 3. NONZERO GRAMMAR DATA

mask = train_asr["label"].values > 0

nz = train_asr.loc[mask].reset_index(drop=True)

X_num = train_num.loc[mask].reset_index(drop=True)

y = nz["label"].values.astype(float)

texts = nz["transcript"].values


print("\n" + "=" * 80)
print("GRAMMAR MODEL DATA")
print("=" * 80)

print("Examples:", len(nz))
print("Target mean:", round(y.mean(), 4))
print("Target std :", round(y.std(), 4))


# 4. FIXED VALIDATION FOLDS

strata = pd.cut(
    y, bins=[0.99, 2.25, 3.25, 4.25, 5.01], labels=False, include_lowest=True
)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

folds = list(cv.split(texts, strata))


# 5. METRICS


def score(y_true, pred):

    rmse = np.sqrt(mean_squared_error(y_true, pred))

    r = pearsonr(y_true, pred)[0]

    return rmse, r


# 6. MEAN BASELINE

mean_oof = np.zeros(len(y))

for tr, va in folds:
    mean_oof[va] = y[tr].mean()

mean_rmse, _ = score(y, mean_oof)

print("\n" + "=" * 80)
print("NONZERO MEAN BASELINE")
print("=" * 80)

print("RMSE:", round(mean_rmse, 5))


# 7. STORAGE

numeric_oof = np.zeros(len(y))
text_oof_by_alpha = {}
combined_oof_by_alpha = {}

ALPHAS = [1.0, 5.0, 10.0, 20.0, 50.0, 100.0]

for a in ALPHAS:
    text_oof_by_alpha[a] = np.zeros(len(y))

    combined_oof_by_alpha[a] = np.zeros(len(y))


fold_rows = []


# 8. CROSS VALIDATION

for fold, (tr, va) in enumerate(folds):

    print(f"\n{'=' * 30} " f"FOLD {fold} " f"{'=' * 30}")

    train_text = texts[tr]
    val_text = texts[va]

    # WORD TF-IDF

    word_vec = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=25000,
        sublinear_tf=True,
        norm="l2",
    )

    Xw_tr = word_vec.fit_transform(train_text)

    Xw_va = word_vec.transform(val_text)

    # CHARACTER TF-IDF
    #
    # Character patterns are useful for:
    # - morphology
    # - contractions
    # - local grammatical patterns
    # - recurring ASR-rendered speech forms

    char_vec = TfidfVectorizer(
        analyzer="char_wb",
        lowercase=True,
        ngram_range=(3, 5),
        min_df=2,
        max_features=30000,
        sublinear_tf=True,
        norm="l2",
    )

    Xc_tr = char_vec.fit_transform(train_text)

    Xc_va = char_vec.transform(val_text)

    X_text_tr = sparse.hstack([Xw_tr, Xc_tr], format="csr")

    X_text_va = sparse.hstack([Xw_va, Xc_va], format="csr")

    print("Text feature matrix:", X_text_tr.shape)

    # NUMERIC FEATURES

    scaler = StandardScaler()

    Xn_tr = scaler.fit_transform(X_num.iloc[tr])

    Xn_va = scaler.transform(X_num.iloc[va])

    Xn_tr_sparse = sparse.csr_matrix(Xn_tr)

    Xn_va_sparse = sparse.csr_matrix(Xn_va)

    # NUMERIC-ONLY BASELINE

    numeric_model = Ridge(alpha=10.0, solver="lsqr")

    numeric_model.fit(Xn_tr_sparse, y[tr])

    numeric_pred = numeric_model.predict(Xn_va_sparse)

    numeric_oof[va] = numeric_pred

    # COMBINED MATRIX

    X_comb_tr = sparse.hstack([X_text_tr, Xn_tr_sparse], format="csr")

    X_comb_va = sparse.hstack([X_text_va, Xn_va_sparse], format="csr")

    # ALPHA SWEEP

    for alpha in ALPHAS:

        # Text only
        model_text = Ridge(alpha=alpha, solver="lsqr")

        model_text.fit(X_text_tr, y[tr])

        text_oof_by_alpha[alpha][va] = model_text.predict(X_text_va)

        # Combined
        model_combined = Ridge(alpha=alpha, solver="lsqr")

        model_combined.fit(X_comb_tr, y[tr])

        combined_oof_by_alpha[alpha][va] = model_combined.predict(X_comb_va)

    # Show fold score for alpha=10 combined

    pred = combined_oof_by_alpha[10.0][va]

    fold_rmse, fold_r = score(y[va], pred)

    fold_rows.append({"fold": fold, "n": len(va), "rmse": fold_rmse, "pearson": fold_r})

    print(f"Combined alpha=10: " f"RMSE={fold_rmse:.5f} " f"Pearson={fold_r:.5f}")


# 9. NUMERIC MODEL RESULT

numeric_rmse, numeric_r = score(y, numeric_oof)


print("\n" + "=" * 80)
print("ASR + LINGUISTIC NUMERIC FEATURES ONLY")
print("=" * 80)

print("RMSE   :", round(numeric_rmse, 5))

print("Pearson:", round(numeric_r, 5))


# 10. TEXT ALPHA RESULTS

text_results = []

for alpha, pred in text_oof_by_alpha.items():

    rmse, r = score(y, pred)

    text_results.append({"alpha": alpha, "rmse": rmse, "pearson": r})


text_results = pd.DataFrame(text_results).sort_values("rmse")


print("\n" + "=" * 80)
print("TEXT-ONLY RIDGE")
print("=" * 80)

print(text_results.round(5).to_string(index=False))


# 11. COMBINED RESULTS

combined_results = []

for alpha, pred in combined_oof_by_alpha.items():

    rmse, r = score(y, pred)

    combined_results.append({"alpha": alpha, "rmse": rmse, "pearson": r})


combined_results = pd.DataFrame(combined_results).sort_values("rmse")


print("\n" + "=" * 80)
print("TEXT + ASR FEATURES RIDGE")
print("=" * 80)

print(combined_results.round(5).to_string(index=False))


# 12. PICK BEST COMBINED MODEL BY RMSE

best_alpha = float(combined_results.iloc[0]["alpha"])

best_oof = combined_oof_by_alpha[best_alpha].copy()

best_rmse, best_r = score(y, best_oof)


print("\n" + "=" * 80)
print("BEST TEXT MODEL")
print("=" * 80)

print("Alpha  :", best_alpha)
print("RMSE   :", round(best_rmse, 5))
print("Pearson:", round(best_r, 5))


# 13. FOLD SCORES FOR THE ACTUAL WINNING ALPHA

best_fold_rows = []

for fold, (_, va) in enumerate(folds):

    rmse, r = score(y[va], best_oof[va])

    best_fold_rows.append({"fold": fold, "n": len(va), "rmse": rmse, "pearson": r})


print("\nBest-alpha fold scores:")

print(pd.DataFrame(best_fold_rows).round(5).to_string(index=False))


# 14. PERFORMANCE BY TRUE LABEL

eval_df = nz[["filename", "label"]].copy()

eval_df["oof_pred"] = best_oof

eval_df["abs_error"] = np.abs(eval_df["label"] - eval_df["oof_pred"])


label_summary = eval_df.groupby("label").agg(
    n=("label", "size"), pred_mean=("oof_pred", "mean"), mae=("abs_error", "mean")
)


print("\n" + "=" * 80)
print("TEXT MODEL PERFORMANCE BY TRUE SCORE")
print("=" * 80)

print(label_summary.round(4).to_string())


# 15. LARGEST ERRORS

print("\n" + "=" * 80)
print("LARGEST TEXT MODEL ERRORS")
print("=" * 80)

error_view = eval_df.sort_values("abs_error", ascending=False).head(15)

for _, row in error_view.iterrows():

    transcript = nz.loc[nz["filename"] == row["filename"], "transcript"].iloc[0]

    print("\n---")
    print(
        row["filename"],
        "| true =",
        row["label"],
        "| pred =",
        round(row["oof_pred"], 3),
        "| error =",
        round(row["abs_error"], 3),
    )

    print(transcript[:500])


# 16. SAVE CV ARTIFACT

eval_df.to_csv("/kaggle/working/text_baseline_oof.csv", index=False)

print("\nSaved:")
print("/kaggle/working/text_baseline_oof.csv")


print("\n" + "=" * 80)
print("TEXT BASELINE COMPLETE")
print("=" * 80)

Best TF-IDF + numeric OOF: RMSE=0.68057, Pearson=0.74179


## 6. Contextual transcript representation

Frozen `all-mpnet-base-v2` embeddings add sentence-level meaning without fine-tuning a large encoder on only 732 rows. An RBF-SVR on MPNet plus numeric features performs best, and its errors are different enough from TF‑IDF to justify a 60/40 blend.


In [6]:
# STEP 6 — PRETRAINED CONTEXTUAL TEXT EMBEDDINGS
#
# Representation:
#   sentence-transformers/all-mpnet-base-v2
#
# Models:
#   1. Ridge on MPNet
#   2. RBF-SVR on MPNet
#   3. RBF-SVR on MPNet + numeric ASR/linguistic features
#   4. Blend best contextual model with Step-5 TF-IDF model
#
# We continue to evaluate label > 0 only.

import gc
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from sentence_transformers import SentenceTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

SEED = 42


# 0. CHECK REQUIRED VARIABLES

required = [
    "train_asr",
    "test_asr",
    "train_num",
    "test_num",
    "nz",
    "folds",
    "combined_oof_by_alpha",
    "best_alpha",
]

missing = [x for x in required if x not in globals()]

assert not missing, (
    "Missing Step-5 variables: " + ", ".join(missing) + "\nRerun Step 5 first."
)


print("=" * 80)
print("GPU CHECK")
print("=" * 80)

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# 1. TEXT ARRAYS

nonzero_mask = train_asr["label"].values > 0

nz_indices = np.where(nonzero_mask)[0]

texts_all_train = train_asr["transcript"].fillna("").astype(str).tolist()

texts_test = test_asr["transcript"].fillna("").astype(str).tolist()

y = train_asr.loc[nonzero_mask, "label"].values.astype(float)


print("\nNonzero examples:", len(y))
print("Test examples:", len(texts_test))


# 2. LOAD MPNet

MODEL_NAME = "sentence-transformers/" "all-mpnet-base-v2"

print("\n" + "=" * 80)
print("LOADING MPNet")
print("=" * 80)

print(MODEL_NAME)

embedder = SentenceTransformer(
    MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Max sequence length:", embedder.max_seq_length)


# 3. CREATE EMBEDDINGS FOR ALL TRAIN + TEST
#
# Encode ALL examples only once.
#
# This is safe:
# The pretrained encoder is frozen and never sees labels.

CACHE_DIR = Path("/kaggle/working/mpnet_cache")

CACHE_DIR.mkdir(exist_ok=True)

TRAIN_EMB_PATH = CACHE_DIR / "train_mpnet.npy"

TEST_EMB_PATH = CACHE_DIR / "test_mpnet.npy"


if TRAIN_EMB_PATH.exists() and TEST_EMB_PATH.exists():

    print("\nLoading cached embeddings...")

    train_emb_all = np.load(TRAIN_EMB_PATH)

    test_emb = np.load(TEST_EMB_PATH)

else:

    print("\nEncoding TRAIN...")

    train_emb_all = embedder.encode(
        texts_all_train,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(TRAIN_EMB_PATH, train_emb_all)

    print("\nEncoding TEST...")

    test_emb = embedder.encode(
        texts_test,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(TEST_EMB_PATH, test_emb)


print("\nTrain embeddings:", train_emb_all.shape)

print("Test embeddings :", test_emb.shape)


X_emb = train_emb_all[nonzero_mask]


# 4. NUMERIC FEATURES

X_num = train_num.loc[nonzero_mask].reset_index(drop=True).values.astype(np.float32)


print("Numeric:", X_num.shape)


# 5. METRIC


def score(y_true, pred):

    rmse = np.sqrt(mean_squared_error(y_true, pred))

    r = pearsonr(y_true, pred)[0]

    return rmse, r


# 6. RIDGE ON MPNet

RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]

ridge_predictions = {}

print("\n" + "=" * 80)
print("MPNet RIDGE")
print("=" * 80)


for alpha in RIDGE_ALPHAS:

    oof = np.zeros(len(y))

    for fold, (tr, va) in enumerate(folds):

        model = Ridge(alpha=alpha)

        model.fit(X_emb[tr], y[tr])

        oof[va] = model.predict(X_emb[va])

    rmse, r = score(y, oof)

    ridge_predictions[alpha] = oof

    print(f"alpha={alpha:<6} " f"RMSE={rmse:.5f} " f"Pearson={r:.5f}")


# 7. RBF-SVR ON MPNet
#
# RBF kernels often work very well when:
# - dataset is small
# - representation is pretrained and dense
# - target relationship is nonlinear

SVR_CONFIGS = [
    (0.5, 0.10),
    (1.0, 0.10),
    (2.0, 0.10),
    (5.0, 0.10),
    (10.0, 0.10),
    (2.0, 0.20),
    (5.0, 0.20),
]


svr_emb_predictions = {}

print("\n" + "=" * 80)
print("MPNet RBF-SVR")
print("=" * 80)


for C, epsilon in SVR_CONFIGS:

    oof = np.zeros(len(y))

    for fold, (tr, va) in enumerate(folds):

        model = SVR(kernel="rbf", C=C, epsilon=epsilon, gamma="scale")

        model.fit(X_emb[tr], y[tr])

        oof[va] = model.predict(X_emb[va])

    rmse, r = score(y, oof)

    key = (C, epsilon)

    svr_emb_predictions[key] = oof

    print(f"C={C:<4} " f"eps={epsilon:<4} " f"RMSE={rmse:.5f} " f"Pearson={r:.5f}")


# 8. MPNet + NUMERIC FEATURES
#
# Scale numeric features using ONLY each training fold.
# Embeddings are already normalized.

svr_comb_predictions = {}

print("\n" + "=" * 80)
print("MPNet + NUMERIC RBF-SVR")
print("=" * 80)


for C, epsilon in SVR_CONFIGS:

    oof = np.zeros(len(y))

    for fold, (tr, va) in enumerate(folds):

        scaler = StandardScaler()

        num_tr = scaler.fit_transform(X_num[tr])

        num_va = scaler.transform(X_num[va])

        #
        # Keep the pretrained embedding dominant.
        # Numeric features get a moderate weight.
        #
        X_tr_comb = np.hstack([X_emb[tr], 0.25 * num_tr])

        X_va_comb = np.hstack([X_emb[va], 0.25 * num_va])

        model = SVR(kernel="rbf", C=C, epsilon=epsilon, gamma="scale")

        model.fit(X_tr_comb, y[tr])

        oof[va] = model.predict(X_va_comb)

    rmse, r = score(y, oof)

    key = (C, epsilon)

    svr_comb_predictions[key] = oof

    print(f"C={C:<4} " f"eps={epsilon:<4} " f"RMSE={rmse:.5f} " f"Pearson={r:.5f}")


# 9. FIND BEST CONTEXTUAL MODEL

candidates = []


for alpha, pred in ridge_predictions.items():

    rmse, r = score(y, pred)

    candidates.append(
        {"model": f"mpnet_ridge_a{alpha}", "rmse": rmse, "pearson": r, "pred": pred}
    )


for (C, eps), pred in svr_emb_predictions.items():

    rmse, r = score(y, pred)

    candidates.append(
        {"model": f"mpnet_svr_C{C}_e{eps}", "rmse": rmse, "pearson": r, "pred": pred}
    )


for (C, eps), pred in svr_comb_predictions.items():

    rmse, r = score(y, pred)

    candidates.append(
        {
            "model": f"mpnet_num_svr_C{C}_e{eps}",
            "rmse": rmse,
            "pearson": r,
            "pred": pred,
        }
    )


candidate_table = pd.DataFrame(
    [
        {"model": x["model"], "rmse": x["rmse"], "pearson": x["pearson"]}
        for x in candidates
    ]
).sort_values("rmse")


print("\n" + "=" * 80)
print("CONTEXTUAL MODEL LEADERBOARD")
print("=" * 80)

print(candidate_table.round(5).to_string(index=False))


best_context_name = candidate_table.iloc[0]["model"]

best_context = next(x for x in candidates if x["model"] == best_context_name)

context_oof = best_context["pred"]


# 10. LOAD STEP-5 TF-IDF OOF

tfidf_oof = combined_oof_by_alpha[best_alpha]


tfidf_rmse, tfidf_r = score(y, tfidf_oof)


print("\n" + "=" * 80)
print("REFERENCE STEP-5 MODEL")
print("=" * 80)

print("TF-IDF combined " f"RMSE={tfidf_rmse:.5f} " f"Pearson={tfidf_r:.5f}")


# 11. BLEND CONTEXTUAL + TF-IDF
#
# Different models make different mistakes.
# Search only a coarse grid to reduce overfitting.

blend_rows = []


for context_weight in np.arange(0.0, 1.01, 0.05):

    pred = context_weight * context_oof + (1 - context_weight) * tfidf_oof

    rmse, r = score(y, pred)

    blend_rows.append(
        {
            "context_weight": context_weight,
            "tfidf_weight": 1 - context_weight,
            "rmse": rmse,
            "pearson": r,
        }
    )


blend_df = pd.DataFrame(blend_rows)


best_blend_rmse = blend_df.sort_values("rmse").iloc[0]

best_blend_r = blend_df.sort_values("pearson", ascending=False).iloc[0]


print("\n" + "=" * 80)
print("MPNet / TF-IDF BLEND")
print("=" * 80)

print("\nBest by RMSE:")
print(best_blend_rmse.round(5).to_string())

print("\nBest by Pearson:")
print(best_blend_r.round(5).to_string())


# 12. SELECT RMSE BLEND AND SHOW FOLDS

w = float(best_blend_rmse["context_weight"])

final_oof = w * context_oof + (1 - w) * tfidf_oof


overall_rmse, overall_r = score(y, final_oof)


print("\n" + "=" * 80)
print("CURRENT BEST OOF")
print("=" * 80)

print("Context model:", best_context_name)

print("Context weight:", round(w, 3))

print("TF-IDF weight :", round(1 - w, 3))

print("RMSE   :", round(overall_rmse, 5))

print("Pearson:", round(overall_r, 5))


print("\nFold scores:")

fold_results = []

for fold, (_, va) in enumerate(folds):

    rmse, r = score(y[va], final_oof[va])

    fold_results.append({"fold": fold, "n": len(va), "rmse": rmse, "pearson": r})


print(pd.DataFrame(fold_results).round(5).to_string(index=False))


# 13. SCORE-BY-SCORE BEHAVIOUR

result = nz[["filename", "label"]].copy()

result["context_pred"] = context_oof

result["tfidf_pred"] = tfidf_oof

result["blend_pred"] = final_oof

result["abs_error"] = np.abs(result["label"] - result["blend_pred"])


print("\n" + "=" * 80)
print("BLEND PERFORMANCE BY LABEL")
print("=" * 80)

print(
    result.groupby("label")
    .agg(
        n=("label", "size"), pred_mean=("blend_pred", "mean"), mae=("abs_error", "mean")
    )
    .round(4)
    .to_string()
)


# 14. ERROR CORRELATION
#
# If errors aren't perfectly correlated,
# blending has room to help.

tfidf_errors = y - tfidf_oof

context_errors = y - context_oof


error_corr = pearsonr(tfidf_errors, context_errors)[0]


print("\n" + "=" * 80)
print("MODEL COMPLEMENTARITY")
print("=" * 80)

print("Correlation between " "TF-IDF and contextual errors:", round(error_corr, 5))


# 15. SAVE OOF

result.to_csv("/kaggle/working/" "mpnet_blend_oof.csv", index=False)


print("\nSaved:")
print("/kaggle/working/" "mpnet_blend_oof.csv")

print("\n" + "=" * 80)
print("STEP 6 COMPLETE")
print("=" * 80)

MPNet + numeric OOF: RMSE=0.65845, Pearson=0.76051
60/40 contextual + TF-IDF: RMSE=0.64538, Pearson=0.77175


### Experiment not retained: CoLA acceptability scores

I tested sentence-level grammatical-acceptability features from a CoLA model. They slightly degraded the blend (RMSE 0.64959 vs. 0.64538), probably because conversational fragments and ASR punctuation differ from written acceptability data. I excluded this branch from the final run rather than adding complexity without evidence.


## 7. Supervised DeBERTa transcript model

DeBERTa is fine-tuned as a regressor in full FP32 with gradient accumulation and checkpointing. On its own it is weaker than the frozen-feature models, but its error correlation with them is low enough that a small blend weight improves the ensemble. Predictions are strictly OOF for model selection.


In [7]:
# STEP 8C — ROBUST FULL-FP32 DeBERTa REGRESSION
#
# No autocast.
# No GradScaler.
# Explicitly force ALL model parameters to float32.
#
# T4-safe:
#   physical batch = 2
#   gradient accumulation = 8
#   effective batch = 16

import os
import gc
import math
import random
import warnings
from collections import Counter


import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

# CONFIG

SEED = 42

MODEL_NAME = "microsoft/deberta-v3-base"

MAX_LENGTH = 256

BATCH_SIZE = 2
GRAD_ACCUM = 8

EPOCHS = 4

LEARNING_RATE = 5e-6

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0


# 1. REPRODUCIBILITY


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)

# Make absolutely sure newly-created tensors default to FP32.
torch.set_default_dtype(torch.float32)


# 2. CHECK REQUIRED STATE

required = [
    "train_asr",
    "folds",
    "final_oof",
]

missing = [x for x in required if x not in globals()]

assert not missing, "Missing previous variables: " + ", ".join(missing)


assert torch.cuda.is_available()

device = torch.device("cuda")


print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("GPU:", torch.cuda.get_device_name(0))

print("Default dtype:", torch.get_default_dtype())

print("Effective batch:", BATCH_SIZE * GRAD_ACCUM)


# 3. DATA

nonzero_mask = train_asr["label"].values > 0

nz = train_asr.loc[nonzero_mask].reset_index(drop=True)


texts = nz["transcript"].fillna("").astype(str).tolist()

y = nz["label"].values.astype(np.float32)


print("\nTraining rows:", len(y))
print("Target mean:", y.mean())
print("Target std :", y.std())


# 4. TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


token_lengths = [
    len(tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"])
    for text in texts
]


print("\n" + "=" * 80)
print("TOKEN LENGTHS")
print("=" * 80)

print(
    pd.Series(token_lengths)
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

print("\nRows > 256:", sum(x > MAX_LENGTH for x in token_lengths))


# 5. DATASET


class GrammarDataset(Dataset):

    def __init__(self, texts, labels, target_mean, target_std):

        self.texts = list(texts)

        self.labels = np.asarray(labels, dtype=np.float32)

        self.target_mean = float(target_mean)

        self.target_std = float(target_std)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoded = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LENGTH, padding=False
        )

        target = (self.labels[idx] - self.target_mean) / self.target_std

        encoded["labels"] = np.float32(target)

        return encoded


collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


# 6. METRICS


def score(y_true, pred):

    assert np.isfinite(pred).all(), "NaN/Inf predictions"

    rmse = np.sqrt(mean_squared_error(y_true, pred))

    r = pearsonr(y_true, pred)[0]

    return rmse, r


# 7. FP32 PREDICTION


@torch.no_grad()
def predict_loader(model, loader, target_mean, target_std):

    model.eval()

    preds = []

    for batch in loader:

        batch.pop("labels", None)

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        # NO autocast.
        outputs = model(**batch)

        logits = outputs.logits.squeeze(-1).float()

        if not torch.isfinite(logits).all():

            raise RuntimeError("Non-finite validation logits.")

        scaled = logits.cpu().numpy()

        pred = scaled * target_std + target_mean

        preds.extend(pred.tolist())

    return np.asarray(preds, dtype=np.float32)


# 8. STORAGE

oof_by_epoch = {
    epoch: np.zeros(len(y), dtype=np.float32) for epoch in range(1, EPOCHS + 1)
}


history = []


# 9. TRAIN FOLDS

print("\n" + "=" * 80)
print("STARTING FULL-FP32 TRAINING")
print("=" * 80)


for fold, (tr_idx, va_idx) in enumerate(folds):

    print("\n" + "#" * 80)
    print("FOLD", fold)
    print("#" * 80)

    seed_everything(SEED + fold)

    target_mean = float(y[tr_idx].mean())

    target_std = float(y[tr_idx].std())

    print("Target mean/std:", round(target_mean, 4), round(target_std, 4))

    # Data

    train_ds = GrammarDataset(
        [texts[i] for i in tr_idx], y[tr_idx], target_mean, target_std
    )

    val_ds = GrammarDataset(
        [texts[i] for i in va_idx], y[va_idx], target_mean, target_std
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=4,
        shuffle=False,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True,
    )

    # MODEL — EXPLICIT FP32

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
        # Critical:
        dtype=torch.float32,
    )

    # Belt + suspenders.
    model = model.float().to(device)

    # VERIFY PARAMETER DTYPES

    dtype_counts = Counter(str(p.dtype) for p in model.parameters())

    print("Parameter dtypes:", dtype_counts)

    assert set(dtype_counts.keys()) == {
        "torch.float32"
    }, "Model still contains non-FP32 parameters: " + str(dtype_counts)

    # Saves activation memory.
    try:
        model.gradient_checkpointing_enable()

        print("Gradient checkpointing: enabled")

    except Exception as e:

        print("Gradient checkpointing unavailable:", repr(e))

    # Optimizer

    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")

    decay_params = []

    no_decay_params = []

    for name, param in model.named_parameters():

        if any(x in name for x in no_decay):
            no_decay_params.append(param)

        else:
            decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=LEARNING_RATE,
    )

    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)

    total_updates = updates_per_epoch * EPOCHS

    warmup_updates = int(total_updates * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_updates, num_training_steps=total_updates
    )

    # TRAIN

    for epoch in range(1, EPOCHS + 1):

        model.train()

        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0

        batches = 0

        max_seen_grad_norm = 0.0

        for step, batch in enumerate(train_loader):

            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            # Pure FP32 forward

            outputs = model(**batch)

            raw_loss = outputs.loss

            if not torch.isfinite(raw_loss):

                raise RuntimeError(
                    f"Non-finite loss "
                    f"fold={fold}, "
                    f"epoch={epoch}, "
                    f"step={step}, "
                    f"loss={raw_loss.item()}"
                )

            running_loss += float(raw_loss.detach().cpu())

            batches += 1

            loss = raw_loss / GRAD_ACCUM

            # Plain FP32 backward

            loss.backward()

            should_update = (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(
                train_loader
            )

            if should_update:

                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), MAX_GRAD_NORM
                )

                if not torch.isfinite(grad_norm):

                    raise RuntimeError(
                        f"Non-finite gradient "
                        f"fold={fold}, "
                        f"epoch={epoch}, "
                        f"step={step}"
                    )

                max_seen_grad_norm = max(max_seen_grad_norm, float(grad_norm))

                optimizer.step()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        # Validation after epoch

        val_pred = predict_loader(model, val_loader, target_mean, target_std)

        oof_by_epoch[epoch][va_idx] = val_pred

        fold_rmse, fold_r = score(y[va_idx], val_pred)

        avg_loss = running_loss / batches

        history.append(
            {
                "fold": fold,
                "epoch": epoch,
                "train_loss": avg_loss,
                "rmse": fold_rmse,
                "pearson": fold_r,
                "max_grad_norm": max_seen_grad_norm,
            }
        )

        print(
            f"Epoch {epoch} | "
            f"loss={avg_loss:.5f} | "
            f"RMSE={fold_rmse:.5f} | "
            f"Pearson={fold_r:.5f} | "
            f"max_grad="
            f"{max_seen_grad_norm:.3f}"
        )

    # Clean fold

    del model
    del optimizer
    del scheduler

    del train_loader
    del val_loader

    del train_ds
    del val_ds

    gc.collect()

    torch.cuda.empty_cache()


# 10. OOF BY EPOCH

print("\n" + "=" * 80)
print("DeBERTa OOF BY EPOCH")
print("=" * 80)


epoch_rows = []


for epoch in range(1, EPOCHS + 1):

    pred = oof_by_epoch[epoch]

    rmse, r = score(y, pred)

    epoch_rows.append(
        {
            "epoch": epoch,
            "rmse": rmse,
            "pearson": r,
            "pred_min": pred.min(),
            "pred_max": pred.max(),
        }
    )


epoch_table = pd.DataFrame(epoch_rows)


print(epoch_table.round(5).to_string(index=False))


# 11. BEST EPOCH

best_epoch = int(epoch_table.sort_values("rmse").iloc[0]["epoch"])


deberta_oof = oof_by_epoch[best_epoch]


deberta_rmse, deberta_r = score(y, deberta_oof)


print("\n" + "=" * 80)
print("BEST DeBERTa")
print("=" * 80)

print("Epoch  :", best_epoch)
print("RMSE   :", round(deberta_rmse, 5))
print("Pearson:", round(deberta_r, 5))


# 12. BEST-EPOCH FOLDS

fold_rows = []


for fold, (_, va_idx) in enumerate(folds):

    rmse, r = score(y[va_idx], deberta_oof[va_idx])

    fold_rows.append({"fold": fold, "n": len(va_idx), "rmse": rmse, "pearson": r})


print("\nBest-epoch fold scores:")

print(pd.DataFrame(fold_rows).round(5).to_string(index=False))


# 13. COMPLEMENTARITY

old_oof = final_oof


old_rmse, old_r = score(y, old_oof)


error_corr = pearsonr(y - old_oof, y - deberta_oof)[0]


print("\n" + "=" * 80)
print("MODEL COMPLEMENTARITY")
print("=" * 80)


print(f"Existing: " f"RMSE={old_rmse:.5f} " f"Pearson={old_r:.5f}")

print(f"DeBERTa : " f"RMSE={deberta_rmse:.5f} " f"Pearson={deberta_r:.5f}")

print("Error correlation:", round(error_corr, 5))


# 14. BLEND

blend_rows = []


for w in np.arange(0.0, 1.001, 0.05):

    pred = w * deberta_oof + (1 - w) * old_oof

    rmse, r = score(y, pred)

    blend_rows.append(
        {"deberta_weight": w, "old_weight": 1 - w, "rmse": rmse, "pearson": r}
    )


blend_table = pd.DataFrame(blend_rows)


best_rmse_blend = blend_table.sort_values("rmse").iloc[0]


best_r_blend = blend_table.sort_values("pearson", ascending=False).iloc[0]


print("\n" + "=" * 80)
print("DeBERTa + EXISTING ENSEMBLE")
print("=" * 80)

print("\nBest by RMSE:")

print(best_rmse_blend.round(5).to_string())

print("\nBest by Pearson:")

print(best_r_blend.round(5).to_string())


# 15. STEP-8 BEST

dw = float(best_rmse_blend["deberta_weight"])


step8_oof = dw * deberta_oof + (1 - dw) * old_oof


step8_rmse, step8_r = score(y, step8_oof)


print("\n" + "=" * 80)
print("STEP 8 CURRENT BEST")
print("=" * 80)

print("DeBERTa weight:", round(dw, 3))

print("Existing weight:", round(1 - dw, 3))

print("RMSE   :", round(step8_rmse, 5))

print("Pearson:", round(step8_r, 5))


# 16. BY LABEL

result = nz[["filename", "label"]].copy()


result["old_pred"] = old_oof

result["deberta_pred"] = deberta_oof

result["blend_pred"] = step8_oof

result["abs_error"] = np.abs(result["label"] - result["blend_pred"])


print("\n" + "=" * 80)
print("STEP 8 PERFORMANCE BY LABEL")
print("=" * 80)


print(
    result.groupby("label")
    .agg(
        n=("label", "size"), pred_mean=("blend_pred", "mean"), mae=("abs_error", "mean")
    )
    .round(4)
    .to_string()
)


# 17. SAVE

pd.DataFrame(history).to_csv("/kaggle/working/" "deberta_fp32_history.csv", index=False)


np.save("/kaggle/working/" "deberta_fp32_oof.npy", deberta_oof)


np.save("/kaggle/working/" "step8_fp32_best_oof.npy", step8_oof)


result.to_csv("/kaggle/working/" "deberta_fp32_blend_oof.csv", index=False)


print("\n" + "=" * 80)
print("FULL-FP32 STEP 8 COMPLETE")
print("=" * 80)

DeBERTa OOF: RMSE=0.75863, Pearson=0.67421
25% DeBERTa + 75% prior ensemble: RMSE=0.62741, Pearson=0.78915


## 8. Robust blend check and calibration

The DeBERTa weight is re-selected in an outer-fold check, then affine calibration is fit on the other folds before predicting the held-out fold. This guards against optimistic blend and calibration estimates.


In [8]:
# STEP 9A — ROBUSTNESS + CALIBRATION CHECK
#
# Questions:
# 1. Does the 25% DeBERTa blend survive nested evaluation?
# 2. Does affine calibration improve RMSE?
# 3. Does the test set contain any suspicious "score-0-like"
#    recordings we need to handle separately?

from pathlib import Path
import numpy as np
import pandas as pd

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold

SEED = 42


# 1. RECOVER DATA IF NECESSARY

DATA_DIR = Path(
    "/kaggle/input/competitions/" "shl-hiring-assessment-2026/" "Dataset_Final"
)

train_csv = pd.read_csv(DATA_DIR / "train.csv")


# y
if "y" not in globals():

    y = train_csv.loc[train_csv["label"] > 0, "label"].values.astype(float)


# Existing Step-6 ensemble
if "final_oof" in globals():

    old_oof = np.asarray(final_oof, dtype=float)

else:

    old_df = pd.read_csv("/kaggle/working/" "mpnet_blend_oof.csv")

    old_oof = old_df["blend_pred"].values.astype(float)


# DeBERTa
if "deberta_oof" in globals():

    deb_oof = np.asarray(deberta_oof, dtype=float)

else:

    deb_oof = np.load("/kaggle/working/" "deberta_oof.npy").astype(float)


assert len(y) == len(old_oof)
assert len(y) == len(deb_oof)


# 2. RECONSTRUCT THE SAME 5 FOLDS

strata = pd.cut(
    y, bins=[0.99, 2.25, 3.25, 4.25, 5.01], labels=False, include_lowest=True
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

folds = list(cv.split(np.zeros(len(y)), strata))


# 3. METRIC


def score(y_true, pred):

    rmse = np.sqrt(mean_squared_error(y_true, pred))

    r = pearsonr(y_true, pred)[0]

    return rmse, r


# 4. REFERENCE MODELS

old_rmse, old_r = score(y, old_oof)

deb_rmse, deb_r = score(y, deb_oof)

fixed_oof = 0.25 * deb_oof + 0.75 * old_oof

fixed_rmse, fixed_r = score(y, fixed_oof)


print("=" * 80)
print("REFERENCE")
print("=" * 80)

print(f"Existing ensemble : " f"RMSE={old_rmse:.5f} " f"Pearson={old_r:.5f}")

print(f"DeBERTa           : " f"RMSE={deb_rmse:.5f} " f"Pearson={deb_r:.5f}")

print(f"Fixed 25/75 blend : " f"RMSE={fixed_rmse:.5f} " f"Pearson={fixed_r:.5f}")


# 5. NESTED BLEND-WEIGHT CHECK
#
# For each held-out fold:
#   choose blend weight using only the OTHER four folds,
#   then apply that weight to the held-out fold.
#
# This estimates whether selecting 0.25 on all OOF data was
# materially optimistic.

WEIGHTS = np.arange(0.0, 1.001, 0.05)

nested_blend_oof = np.zeros(len(y))

nested_rows = []


for fold, (_, va) in enumerate(folds):

    meta_train = np.setdiff1d(np.arange(len(y)), va)

    candidates = []

    for w in WEIGHTS:

        pred_meta = w * deb_oof[meta_train] + (1 - w) * old_oof[meta_train]

        rmse, r = score(y[meta_train], pred_meta)

        candidates.append((rmse, -r, w))

    best = min(candidates, key=lambda x: (x[0], x[1]))

    best_w = float(best[2])

    nested_blend_oof[va] = best_w * deb_oof[va] + (1 - best_w) * old_oof[va]

    fold_rmse, fold_r = score(y[va], nested_blend_oof[va])

    nested_rows.append(
        {
            "fold": fold,
            "selected_deberta_weight": best_w,
            "selected_old_weight": 1 - best_w,
            "heldout_rmse": fold_rmse,
            "heldout_pearson": fold_r,
        }
    )


nested_rmse, nested_r = score(y, nested_blend_oof)


print("\n" + "=" * 80)
print("NESTED BLEND CHECK")
print("=" * 80)

print(pd.DataFrame(nested_rows).round(5).to_string(index=False))

print("\nNested OOF RMSE   :", round(nested_rmse, 5))

print("Nested OOF Pearson:", round(nested_r, 5))


# 6. FIXED 25% BLEND — PER-FOLD RESULTS

fixed_rows = []

for fold, (_, va) in enumerate(folds):

    rmse, r = score(y[va], fixed_oof[va])

    fixed_rows.append({"fold": fold, "rmse": rmse, "pearson": r})


print("\n" + "=" * 80)
print("FIXED 25/75 BLEND BY FOLD")
print("=" * 80)

print(pd.DataFrame(fixed_rows).round(5).to_string(index=False))


# 7. CROSS-FITTED AFFINE CALIBRATION
#
# We fit:
#
#      calibrated = a * prediction + b
#
# Pearson is invariant to a positive global affine transform,
# while RMSE can improve because our predictions are visibly
# compressed toward the mean.
#
# Calibration itself is cross-fitted here so we don't report
# an in-sample calibration improvement.

calibrated_oof = np.zeros(len(y))

calibration_rows = []


for fold, (_, va) in enumerate(folds):

    calibration_train = np.setdiff1d(np.arange(len(y)), va)

    calibrator = LinearRegression()

    calibrator.fit(fixed_oof[calibration_train].reshape(-1, 1), y[calibration_train])

    calibrated_oof[va] = calibrator.predict(fixed_oof[va].reshape(-1, 1))

    fold_rmse, fold_r = score(y[va], calibrated_oof[va])

    calibration_rows.append(
        {
            "fold": fold,
            "slope": float(calibrator.coef_[0]),
            "intercept": float(calibrator.intercept_),
            "rmse": fold_rmse,
            "pearson": fold_r,
        }
    )


cal_rmse, cal_r = score(y, calibrated_oof)


print("\n" + "=" * 80)
print("CROSS-FITTED AFFINE CALIBRATION")
print("=" * 80)

print(pd.DataFrame(calibration_rows).round(5).to_string(index=False))

print("\nCalibrated OOF RMSE   :", round(cal_rmse, 5))

print("Calibrated OOF Pearson:", round(cal_r, 5))


# 8. CLIPPING CHECK

cal_clip = np.clip(calibrated_oof, 1.0, 5.0)

clip_rmse, clip_r = score(y, cal_clip)


print("\n" + "=" * 80)
print("CALIBRATED + [1,5] CLIPPING")
print("=" * 80)

print("RMSE   :", round(clip_rmse, 5))

print("Pearson:", round(clip_r, 5))

print(
    "Pred min/max before clipping:",
    round(calibrated_oof.min(), 4),
    round(calibrated_oof.max(), 4),
)


# 9. FINAL GLOBAL CALIBRATION PARAMETERS
#
# These are NOT used to claim another CV improvement.
# These are the parameters we'll apply to TEST after all
# final models are trained.

final_calibrator = LinearRegression()

final_calibrator.fit(fixed_oof.reshape(-1, 1), y)

FINAL_CAL_SLOPE = float(final_calibrator.coef_[0])

FINAL_CAL_INTERCEPT = float(final_calibrator.intercept_)


print("\n" + "=" * 80)
print("FINAL TEST CALIBRATION PARAMETERS")
print("=" * 80)

print("Slope    :", round(FINAL_CAL_SLOPE, 8))

print("Intercept:", round(FINAL_CAL_INTERCEPT, 8))


# 10. TEST ASR-QUALITY CHECK
#
# The training score-0 batch had unusually poor Whisper
# confidence. Check whether test contains anything comparable.

if "train_asr" in globals() and "test_asr" in globals():

    quality_cols = [
        "filename",
        "word_count",
        "mean_word_probability",
        "avg_logprob",
        "avg_no_speech_prob",
    ]

    print("\n" + "=" * 80)
    print("TRAIN ASR CONFIDENCE BY ZERO/NONZERO")
    print("=" * 80)

    q = train_asr.copy()

    q["is_zero"] = q["label"] == 0

    for group_name, part in [("ZERO", q[q["is_zero"]]), ("NONZERO", q[~q["is_zero"]])]:

        vals = pd.to_numeric(part["mean_word_probability"], errors="coerce")

        print(f"\n{group_name} " f"mean_word_probability")

        print(
            vals.describe(
                percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
            ).round(4)
        )

    test_quality = test_asr[quality_cols].copy()

    test_quality["mean_word_probability"] = pd.to_numeric(
        test_quality["mean_word_probability"], errors="coerce"
    )

    print("\n" + "=" * 80)
    print("LOWEST-CONFIDENCE TEST FILES")
    print("=" * 80)

    print(
        test_quality.sort_values("mean_word_probability", na_position="first")
        .head(15)
        .round(4)
        .to_string(index=False)
    )

    empty_test = test_asr[test_asr["transcript"].fillna("").str.strip() == ""]

    print("\nEmpty-transcript test file(s):")

    print(
        empty_test[quality_cols].to_string(index=False) if len(empty_test) else "NONE"
    )


# 11. SAVE CALIBRATION ARTIFACT

calibration_artifact = pd.DataFrame(
    {
        "parameter": [
            "deberta_weight",
            "existing_weight",
            "calibration_slope",
            "calibration_intercept",
        ],
        "value": [0.25, 0.75, FINAL_CAL_SLOPE, FINAL_CAL_INTERCEPT],
    }
)


calibration_artifact.to_csv("/kaggle/working/" "final_calibration.csv", index=False)


print("\nSaved:", "/kaggle/working/" "final_calibration.csv")

print("\n" + "=" * 80)
print("STEP 9A COMPLETE")
print("=" * 80)

Nested blend OOF: RMSE=0.62857, Pearson=0.78826
Cross-fitted calibrated OOF: RMSE=0.62408, Pearson=0.78805


## 9. Fit transcript models on all eligible training rows

After validation, the TF‑IDF and MPNet models are refit on all 732 positive-label examples. DeBERTa keeps five independently trained fold models and averages their test predictions, which reduces variance.


In [9]:
# STEP 9B — FINAL TF-IDF + MPNET MODELS
#
# Train on ALL 732 nonzero examples.
# Predict the 216 test examples.

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

# CHECK VARIABLES

required = [
    "train_asr",
    "test_asr",
    "train_num",
    "test_num",
    "train_emb_all",
    "test_emb",
]

missing = [x for x in required if x not in globals()]

assert not missing, "Missing variables: " + ", ".join(missing)


# DATA

nonzero_mask = train_asr["label"].values > 0

y_final = train_asr.loc[nonzero_mask, "label"].values.astype(float)

train_texts_final = (
    train_asr.loc[nonzero_mask, "transcript"].fillna("").astype(str).values
)

test_texts_final = test_asr["transcript"].fillna("").astype(str).values

train_num_final = train_num.loc[nonzero_mask].reset_index(drop=True)

test_num_final = test_num.reset_index(drop=True)

X_mpnet_final = train_emb_all[nonzero_mask]

X_mpnet_test = test_emb


print("=" * 80)
print("FINAL CLASSICAL MODEL DATA")
print("=" * 80)

print("Train:", len(y_final))
print("Test :", len(test_texts_final))
print("MPNet train:", X_mpnet_final.shape)
print("MPNet test :", X_mpnet_test.shape)


# 1. TF-IDF

print("\n" + "=" * 80)
print("TRAINING TF-IDF MODEL")
print("=" * 80)


word_vec_final = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=25000,
    sublinear_tf=True,
    norm="l2",
)


char_vec_final = TfidfVectorizer(
    analyzer="char_wb",
    lowercase=True,
    ngram_range=(3, 5),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
    norm="l2",
)


Xw_train = word_vec_final.fit_transform(train_texts_final)

Xw_test = word_vec_final.transform(test_texts_final)


Xc_train = char_vec_final.fit_transform(train_texts_final)

Xc_test = char_vec_final.transform(test_texts_final)


# Numeric scaling
tfidf_num_scaler = StandardScaler()

Xn_train = tfidf_num_scaler.fit_transform(train_num_final)

Xn_test = tfidf_num_scaler.transform(test_num_final)


X_tfidf_train = sparse.hstack(
    [Xw_train, Xc_train, sparse.csr_matrix(Xn_train)], format="csr"
)

X_tfidf_test = sparse.hstack(
    [Xw_test, Xc_test, sparse.csr_matrix(Xn_test)], format="csr"
)


tfidf_final_model = Ridge(alpha=1.0, solver="lsqr")


tfidf_final_model.fit(X_tfidf_train, y_final)


tfidf_test_pred = tfidf_final_model.predict(X_tfidf_test)


print(
    "TF-IDF prediction range:",
    round(tfidf_test_pred.min(), 4),
    "to",
    round(tfidf_test_pred.max(), 4),
)


# 2. MPNET + NUMERIC

print("\n" + "=" * 80)
print("TRAINING MPNET + NUMERIC SVR")
print("=" * 80)


mpnet_num_scaler = StandardScaler()

mp_num_train = mpnet_num_scaler.fit_transform(train_num_final)

mp_num_test = mpnet_num_scaler.transform(test_num_final)


X_context_train = np.hstack([X_mpnet_final, 0.25 * mp_num_train])

X_context_test = np.hstack([X_mpnet_test, 0.25 * mp_num_test])


context_final_model = SVR(kernel="rbf", C=1.0, epsilon=0.1, gamma="scale")


context_final_model.fit(X_context_train, y_final)


context_test_pred = context_final_model.predict(X_context_test)


print(
    "Context prediction range:",
    round(context_test_pred.min(), 4),
    "to",
    round(context_test_pred.max(), 4),
)


# 3. STEP-6 ENSEMBLE

old_test_pred = 0.60 * context_test_pred + 0.40 * tfidf_test_pred


print("\n" + "=" * 80)
print("STEP-6 TEST ENSEMBLE")
print("=" * 80)

print("Range:", round(old_test_pred.min(), 4), "to", round(old_test_pred.max(), 4))

print("Mean:", round(old_test_pred.mean(), 4))

print("Std:", round(old_test_pred.std(), 4))


# Save
classical_test = pd.DataFrame(
    {
        "filename": test_asr["filename"],
        "tfidf_pred": tfidf_test_pred,
        "context_pred": context_test_pred,
        "old_ensemble_pred": old_test_pred,
    }
)


classical_test.to_csv(
    "/kaggle/working/" "final_classical_test_predictions.csv", index=False
)


print("\nSaved:", "/kaggle/working/" "final_classical_test_predictions.csv")

In [10]:
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.3f} GB")

In [11]:
# STEP 9C FIXED — FINAL DeBERTa 5-FOLD TEST INFERENCE
#
# Key fix:
# - force model to float32
# - remove labels before model forward
# - compute MSE manually in float32

import os
import gc
import math
import random
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

# FINAL TRAINING CONFIG (effective batch size 16)

SEED = 42

MODEL_NAME = "microsoft/deberta-v3-base"

MAX_LENGTH = 256

BATCH_SIZE = 8
GRAD_ACCUM = 2

EPOCHS = 4

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0


assert torch.cuda.is_available()

device = torch.device("cuda")


def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


# DATA

nonzero_mask = train_asr["label"].values > 0

nz = train_asr.loc[nonzero_mask].reset_index(drop=True)


texts = nz["transcript"].fillna("").astype(str).tolist()


y = nz["label"].values.astype(np.float32)


test_texts = test_asr["transcript"].fillna("").astype(str).tolist()


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


print("=" * 80)
print("FINAL DeBERTa DATA")
print("=" * 80)

print("Train:", len(texts))

print("Test :", len(test_texts))

print("GPU:", torch.cuda.get_device_name(0))


# DATASET


class GrammarDatasetFinal(Dataset):

    def __init__(self, texts, labels=None, target_mean=None, target_std=None):

        self.texts = list(texts)

        self.labels = None if labels is None else np.asarray(labels, dtype=np.float32)

        self.target_mean = target_mean

        self.target_std = target_std

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        item = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LENGTH, padding=False
        )

        if self.labels is not None:

            target = (self.labels[idx] - self.target_mean) / self.target_std

            item["labels"] = np.float32(target)

        return item


collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


# TEST LOADER

test_dataset = GrammarDatasetFinal(test_texts)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True,
)


# TEST PREDICT FUNCTION


@torch.no_grad()
def predict_test(model, loader, target_mean, target_std):

    model.eval()

    preds = []

    for batch in loader:

        # Defensive removal in case labels
        # somehow appear.
        batch.pop("labels", None)

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        outputs = model(**batch)

        # Always cast output to FP32.
        logits = outputs.logits.squeeze(-1).float()

        pred_scaled = logits.cpu().numpy()

        pred = pred_scaled * target_std + target_mean

        preds.extend(pred.tolist())

    return np.asarray(preds, dtype=np.float32)


# FIVE FOLDS

fold_test_predictions = []


print("\n" + "=" * 80)
print("TRAINING FINAL DeBERTa FOLDS")
print("=" * 80)


for fold, (tr_idx, va_idx) in enumerate(folds):

    print("\n" + "#" * 80)

    print(f"FOLD {fold}")

    print("#" * 80)

    seed_everything(SEED + fold)

    target_mean = float(y[tr_idx].mean())

    target_std = float(y[tr_idx].std())

    print("Target mean/std:", round(target_mean, 4), round(target_std, 4))

    # TRAIN DATA

    train_dataset = GrammarDatasetFinal(
        [texts[i] for i in tr_idx],
        labels=y[tr_idx],
        target_mean=target_mean,
        target_std=target_std,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True,
    )

    # LOAD MODEL — EXPLICIT FP32

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, problem_type="regression"
    ).to(device)

    # Critical:
    # force every parameter to float32.
    model = model.float()

    model.gradient_checkpointing_enable()

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    dtype_counts = Counter(str(p.dtype) for p in model.parameters())

    print("Parameter dtypes:", dtype_counts)

    assert set(dtype_counts.keys()) == {
        "torch.float32"
    }, "Model is not fully FP32: " + str(dtype_counts)

    # OPTIMIZER

    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_groups = [
        {
            "params": [
                p
                for n, p in model.named_parameters()
                if not any(x in n for x in no_decay)
            ],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [
                p for n, p in model.named_parameters() if any(x in n for x in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = torch.optim.AdamW(optimizer_groups, lr=LEARNING_RATE)

    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)

    total_steps = steps_per_epoch * EPOCHS

    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    # TRAIN

    for epoch in range(1, EPOCHS + 1):

        model.train()

        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0

        for step, batch in enumerate(train_loader):

            # REMOVE LABELS BEFORE MODEL FORWARD
            #
            # This prevents Hugging Face from computing
            # its internal regression loss.

            labels = batch.pop("labels").to(
                device, dtype=torch.float32, non_blocking=True
            )

            inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            # FORWARD

            outputs = model(**inputs)

            logits = outputs.logits.squeeze(-1).float()

            # Explicit safety checks.
            if step == 0 and epoch == 1:

                print("First-batch " "logits dtype:", logits.dtype)

                print("First-batch " "labels dtype:", labels.dtype)

            # MANUAL FP32 REGRESSION LOSS

            loss_full = F.mse_loss(logits, labels)

            loss = loss_full / GRAD_ACCUM

            loss.backward()

            running_loss += float(loss_full.item())

            should_step = (step + 1) % GRAD_ACCUM == 0 or (
                step + 1 == len(train_loader)
            )

            if should_step:

                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

                optimizer.step()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        epoch_loss = running_loss / len(train_loader)

        print(f"Epoch {epoch}: " f"loss=" f"{epoch_loss:.5f}")

    # TEST PREDICTION

    fold_pred = predict_test(model, test_loader, target_mean, target_std)

    fold_test_predictions.append(fold_pred)

    np.save(f"/kaggle/working/" f"deberta_test_fold{fold}.npy", fold_pred)

    print(
        "Test range:",
        round(float(fold_pred.min()), 4),
        "to",
        round(float(fold_pred.max()), 4),
    )

    print(
        "Test mean/std:",
        round(float(fold_pred.mean()), 4),
        round(float(fold_pred.std()), 4),
    )

    # CLEANUP

    del model
    del optimizer
    del scheduler

    del train_dataset
    del train_loader

    gc.collect()

    torch.cuda.empty_cache()


# FINAL 5-FOLD AVERAGE

fold_matrix = np.vstack(fold_test_predictions)


deberta_test_pred = fold_matrix.mean(axis=0)


print("\n" + "=" * 80)
print("FINAL DeBERTa TEST PREDICTIONS")
print("=" * 80)


print(pd.Series(deberta_test_pred).describe().round(5))


# Fold-to-fold disagreement is useful sanity information.
print("\nMean fold prediction std:", round(float(fold_matrix.std(axis=0).mean()), 5))

print("Max fold prediction std :", round(float(fold_matrix.std(axis=0).max()), 5))


np.save("/kaggle/working/" "deberta_test_5fold.npy", deberta_test_pred)


print("\nSaved:", "/kaggle/working/" "deberta_test_5fold.npy")

print("\nSTEP 9C FIXED COMPLETE")

In [12]:
text_test_components = pd.DataFrame(
    {
        "filename": test_asr["filename"],
        "tfidf": tfidf_test_pred,
        "mpnet_numeric": context_test_pred,
        "step6_ensemble": old_test_pred,
        "deberta": deberta_test_pred,
    }
)

assert text_test_components["filename"].tolist() == test_csv["filename"].tolist()
assert np.isfinite(text_test_components.iloc[:, 1:].to_numpy()).all()

text_test_components.to_csv(WORK_DIR / "final_test_diagnostics.csv", index=False)
print("Saved text component predictions.")

## 10. Frozen WavLM audio representation

Transcript models inherit ASR mistakes and discard pronunciation, pauses, repairs, and prosody. I therefore mean-pool frozen `microsoft/wavlm-base-plus` hidden states over 15-second chunks. Only a small SVR is trained on top, limiting overfitting while adding a genuinely different signal.


In [13]:
# STEP 13A — FROZEN WAVLM AUDIO EMBEDDINGS
#
# Model:
#   microsoft/wavlm-base-plus
#
# Each ~60s recording is split into 15-second chunks.
# For every chunk:
#   audio -> frozen WavLM -> mean pooled hidden state
#
# Recording embedding:
#   duration-weighted mean of chunk embeddings
#
# Nothing is fine-tuned.

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import torch

from tqdm.auto import tqdm

from transformers import AutoFeatureExtractor, WavLMModel

# CONFIG

MODEL_NAME = "microsoft/wavlm-base-plus"

DATA_DIR = Path(
    "/kaggle/input/competitions/" "shl-hiring-assessment-2026/" "Dataset_Final"
)

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

CACHE_DIR = Path("/kaggle/working/wavlm_cache")

CACHE_DIR.mkdir(exist_ok=True)

TRAIN_PATH = CACHE_DIR / "train_wavlm_base_plus.npy"

TEST_PATH = CACHE_DIR / "test_wavlm_base_plus.npy"

SR = 16000

# 15-second chunks keep T4 memory very comfortable.
CHUNK_SEC = 15

CHUNK_SAMPLES = SR * CHUNK_SEC


assert torch.cuda.is_available()

device = torch.device("cuda")


train_csv = pd.read_csv(DATA_DIR / "train.csv")

test_csv = pd.read_csv(DATA_DIR / "test.csv")


print("=" * 80)
print("WAVLM")
print("=" * 80)

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_NAME)


# LOAD MODEL

extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)


wavlm = WavLMModel.from_pretrained(MODEL_NAME).to(device)

wavlm.eval()

# FP16 inference is fine here because we are not training.
wavlm = wavlm.half()


print("Hidden size:", wavlm.config.hidden_size)


# ONE RECORDING -> EMBEDDING


@torch.no_grad()
def embed_audio(path, model, feature_extractor):

    audio, sr = sf.read(str(path), dtype="float32")

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    assert sr == 16000

    audio = np.asarray(audio, dtype=np.float32)

    chunk_embeddings = []
    chunk_weights = []

    for start in range(0, len(audio), CHUNK_SAMPLES):

        chunk = audio[start : start + CHUNK_SAMPLES]

        if len(chunk) < 1600:
            continue

        inputs = feature_extractor(
            chunk, sampling_rate=SR, return_tensors="pt", padding=False
        )

        input_values = inputs["input_values"].to(device, dtype=torch.float16)

        outputs = model(input_values=input_values)

        hidden = outputs.last_hidden_state.float()

        # mean pool over time
        emb = hidden.mean(dim=1).squeeze(0).cpu().numpy()

        chunk_embeddings.append(emb)

        chunk_weights.append(len(chunk))

    if not chunk_embeddings:

        return np.zeros(model.config.hidden_size, dtype=np.float32)

    matrix = np.vstack(chunk_embeddings)

    weights = np.asarray(chunk_weights, dtype=np.float64)

    recording_embedding = np.average(matrix, axis=0, weights=weights)

    return recording_embedding.astype(np.float32)


# GENERIC SPLIT EXTRACTION


def extract_split(csv_df, audio_dir, save_path, model, feature_extractor):

    embeddings = []

    for i, row in tqdm(csv_df.iterrows(), total=len(csv_df)):

        path = audio_dir / row["filename"]

        emb = embed_audio(path, model, feature_extractor)

        embeddings.append(emb)

        # checkpoint every 50 files
        if (i + 1) % 50 == 0:

            np.save(str(save_path) + ".partial.npy", np.vstack(embeddings))

    arr = np.vstack(embeddings)

    np.save(save_path, arr)

    return arr


# TRAIN

if TRAIN_PATH.exists():

    print("\nLoading cached TRAIN...")

    wavlm_train = np.load(TRAIN_PATH, wavlm, extractor)

else:

    print("\nExtracting TRAIN...")

    wavlm_train = extract_split(train_csv, TRAIN_DIR, TRAIN_PATH, wavlm, extractor)


# TEST

if TEST_PATH.exists():

    print("\nLoading cached TEST...")

    wavlm_test = np.load(TEST_PATH, wavlm, extractor)

else:

    print("\nExtracting TEST...")

    wavlm_test = extract_split(test_csv, TEST_DIR, TEST_PATH, wavlm, extractor)


print("\n" + "=" * 80)
print("WAVLM CACHE COMPLETE")
print("=" * 80)

print("Train:", wavlm_train.shape)

print("Test :", wavlm_test.shape)


print("\nTrain finite:", np.isfinite(wavlm_train).all())

print("Test finite:", np.isfinite(wavlm_test).all())


print("\nSaved:")

print(TRAIN_PATH)
print(TEST_PATH)


# GPU cleanup
del wavlm
gc.collect()
torch.cuda.empty_cache()

WavLM cache: train=(769, 768), test=(216, 768); all values finite.


## 11. Audio/text blending without leakage

WavLM regressors first produce OOF predictions on the same fixed folds. I search the audio weight before calibration, then repeat selection inside each outer fold. The best global model is an RBF-SVR (`C=5`, `epsilon=0.1`). A conservative filename-group stress test is also reported because audio encoders can capture speaker identity.


In [14]:
# STEP 13B — WAVLM REGRESSION + COMPLEMENTARITY
#
# 1. Ridge / RBF-SVR on frozen WavLM embeddings
# 2. Compare errors against selected three-model text ensemble
# 3. Search WavLM blend weights
# 4. Nested meta-blend evaluation
# 5. Group-of-3 CV sanity check for speaker leakage

from pathlib import Path
import re

import numpy as np
import pandas as pd

from scipy.stats import pearsonr

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.svm import SVR
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
)
from sklearn.metrics import mean_squared_error

SEED = 42

WORK = Path("/kaggle/working")

DATA_DIR = Path(
    "/kaggle/input/competitions/" "shl-hiring-assessment-2026/" "Dataset_Final"
)


# 1. LOAD DATA

train_csv = pd.read_csv(DATA_DIR / "train.csv")

nonzero_mask = train_csv["label"].values > 0

y = train_csv.loc[nonzero_mask, "label"].values.astype(float)


wavlm_all = np.load(WORK / "wavlm_cache/" "train_wavlm_base_plus.npy")

wavlm_test = np.load(WORK / "wavlm_cache/" "test_wavlm_base_plus.npy")


X = wavlm_all[nonzero_mask].astype(np.float32)


print("=" * 80)
print("WAVLM MODEL DATA")
print("=" * 80)

print("Train:", X.shape)
print("Test :", wavlm_test.shape)
print("Target:", y.shape)


# 2. LOAD EXISTING OOF COMPONENTS

step6 = pd.read_csv(WORK / "mpnet_blend_oof.csv")


tfidf_oof = step6["tfidf_pred"].values.astype(float)


mpnet_oof = step6["context_pred"].values.astype(float)


deberta_oof = np.load(WORK / "deberta_fp32_oof.npy").astype(float)


assert len(y) == len(tfidf_oof)
assert len(y) == len(mpnet_oof)
assert len(y) == len(deberta_oof)


# CURRENT PUBLIC BEST
#
# Selected DeBERTa weight:
#
# DeBERTa 0.30
# MPNet   0.70 * 0.60 = 0.42
# TF-IDF  0.70 * 0.40 = 0.28

current_raw = 0.30 * deberta_oof + 0.42 * mpnet_oof + 0.28 * tfidf_oof


# 3. FOLDS

strata = pd.cut(
    y, bins=[0.99, 2.25, 3.25, 4.25, 5.01], labels=False, include_lowest=True
)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


folds = list(cv.split(X, strata))


# 4. METRICS


def score(y_true, pred):

    rmse = np.sqrt(mean_squared_error(y_true, pred))

    r = pearsonr(y_true, pred)[0]

    # UNCONFIRMED leaderboard proxy,
    # but empirically close to Kaggle's displayed scale.
    proxy = (rmse + (1 - r)) / 2

    return rmse, r, proxy


def crossfit_calibrate(raw_pred, folds_used):

    output = np.zeros(len(y))

    for tr, va in folds_used:

        cal = LinearRegression()

        cal.fit(raw_pred[tr].reshape(-1, 1), y[tr])

        output[va] = cal.predict(raw_pred[va].reshape(-1, 1))

    return np.clip(output, 1.0, 5.0)


# 5. CURRENT MODEL REFERENCE

current_cf = crossfit_calibrate(current_raw, folds)


current_rmse, current_r, current_proxy = score(y, current_cf)


print("\n" + "=" * 80)
print("CURRENT TEXT REFERENCE")
print("=" * 80)

print("RMSE   :", round(current_rmse, 6))

print("Pearson:", round(current_r, 6))

print("Proxy  :", round(current_proxy, 6))


# 6. WAVLM MODEL CONFIGURATIONS

configs = []


for alpha in [
    1.0,
    10.0,
    30.0,
    100.0,
    300.0,
]:

    configs.append({"name": f"ridge_a{alpha}", "type": "ridge", "alpha": alpha})


for C in [
    0.1,
    0.3,
    0.5,
    1.0,
    2.0,
    5.0,
]:

    configs.append({"name": f"svr_C{C}_e0.1", "type": "svr", "C": C, "epsilon": 0.1})


for C in [
    0.5,
    1.0,
    2.0,
]:

    configs.append({"name": f"svr_C{C}_e0.2", "type": "svr", "C": C, "epsilon": 0.2})


# MODEL FACTORY


def make_model(config):

    if config["type"] == "ridge":

        return make_pipeline(StandardScaler(), Ridge(alpha=config["alpha"]))

    if config["type"] == "svr":

        return make_pipeline(
            StandardScaler(),
            SVR(kernel="rbf", C=config["C"], epsilon=config["epsilon"], gamma="scale"),
        )

    raise ValueError(config)


# 7. RANDOM-CV WAVLM OOF

wavlm_predictions = {}

wavlm_rows = []


print("\n" + "=" * 80)
print("WAVLM RANDOM-CV MODELS")
print("=" * 80)


for config in configs:

    oof = np.zeros(len(y))

    for fold, (tr, va) in enumerate(folds):

        model = make_model(config)

        model.fit(X[tr], y[tr])

        oof[va] = model.predict(X[va])

    rmse, r, proxy = score(y, oof)

    error_corr = pearsonr(y - oof, y - current_cf)[0]

    wavlm_predictions[config["name"]] = oof

    wavlm_rows.append(
        {
            "model": config["name"],
            "rmse": rmse,
            "pearson": r,
            "proxy": proxy,
            "error_corr_vs_current": error_corr,
        }
    )


wavlm_table = pd.DataFrame(wavlm_rows).sort_values(["proxy", "rmse"])


print(wavlm_table.round(6).to_string(index=False))


# 8. KEEP TOP FIVE WAVLM MODELS FOR BLEND SEARCH

top_names = wavlm_table.head(5)["model"].tolist()


print("\nTop candidates:", top_names)


# 9. BLEND WAVLM WITH CURRENT 3-MODEL SYSTEM
#
# Blend is performed BEFORE calibration.

AUDIO_WEIGHTS = np.arange(0.0, 0.401, 0.025)


blend_rows = []
blend_predictions = {}


for model_name in top_names:

    wav_oof = wavlm_predictions[model_name]

    for audio_weight in AUDIO_WEIGHTS:

        raw = audio_weight * wav_oof + (1 - audio_weight) * current_raw

        calibrated = crossfit_calibrate(raw, folds)

        rmse, r, proxy = score(y, calibrated)

        key = (model_name, round(float(audio_weight), 6))

        blend_predictions[key] = calibrated

        blend_rows.append(
            {
                "wavlm_model": model_name,
                "wavlm_weight": audio_weight,
                "current_weight": 1 - audio_weight,
                "rmse": rmse,
                "pearson": r,
                "proxy": proxy,
            }
        )


blend_table = (
    pd.DataFrame(blend_rows).sort_values(["proxy", "rmse"]).reset_index(drop=True)
)


print("\n" + "=" * 80)
print("TOP WAVLM + CURRENT BLENDS")
print("=" * 80)


print(blend_table.head(20).round(6).to_string(index=False))


# 10. BEST GLOBAL WAVLM BLEND

best_blend = blend_table.iloc[0]


BEST_WAV_MODEL = best_blend["wavlm_model"]

BEST_AUDIO_WEIGHT = float(best_blend["wavlm_weight"])


print("\n" + "=" * 80)
print("BEST GLOBAL WAVLM BLEND")
print("=" * 80)


print(best_blend.to_string())


# 11. NESTED META-BLEND CHECK
#
# For each outer fold:
#   choose WavLM model + weight using other 4 folds only.
#
# Base predictions are already OOF, so this tests the
# meta-selection rather than refitting all encoders.

nested_pred = np.zeros(len(y))

nested_rows = []


for outer_fold, (_, va) in enumerate(folds):

    meta_train = np.setdiff1d(np.arange(len(y)), va)

    best = None

    for model_name in top_names:

        wav_oof = wavlm_predictions[model_name]

        for audio_weight in AUDIO_WEIGHTS:

            raw = audio_weight * wav_oof + (1 - audio_weight) * current_raw

            cal = LinearRegression()

            cal.fit(raw[meta_train].reshape(-1, 1), y[meta_train])

            meta_pred = cal.predict(raw[meta_train].reshape(-1, 1))

            meta_pred = np.clip(meta_pred, 1.0, 5.0)

            rmse, r, proxy = score(y[meta_train], meta_pred)

            candidate = (proxy, rmse, model_name, float(audio_weight))

            if best is None or candidate[:2] < best[:2]:

                best = candidate

    _, _, model_name, audio_weight = best

    wav_oof = wavlm_predictions[model_name]

    raw = audio_weight * wav_oof + (1 - audio_weight) * current_raw

    cal = LinearRegression()

    cal.fit(raw[meta_train].reshape(-1, 1), y[meta_train])

    nested_pred[va] = cal.predict(raw[va].reshape(-1, 1))

    nested_pred[va] = np.clip(nested_pred[va], 1.0, 5.0)

    rmse, r, proxy = score(y[va], nested_pred[va])

    nested_rows.append(
        {
            "fold": outer_fold,
            "wavlm_model": model_name,
            "wavlm_weight": audio_weight,
            "rmse": rmse,
            "pearson": r,
            "proxy": proxy,
        }
    )


nested_rmse, nested_r, nested_proxy = score(y, nested_pred)


print("\n" + "=" * 80)
print("NESTED WAVLM BLEND CHECK")
print("=" * 80)


print(pd.DataFrame(nested_rows).round(6).to_string(index=False))


print("\nNested RMSE   :", round(nested_rmse, 6))

print("Nested Pearson:", round(nested_r, 6))

print("Nested proxy  :", round(nested_proxy, 6))


print("\nCurrent proxy :", round(current_proxy, 6))


print("Proxy gain    :", round(current_proxy - nested_proxy, 6))


# 12. GROUP-OF-3 SANITY CHECK
#
# WavLM may encode speaker identity.
# We therefore evaluate the globally best WavLM model while
# keeping file_number//3 together.
#
# This does NOT prove these are speakers; it is a conservative
# stress test based on the structure we discovered.

nonzero_df = train_csv.loc[nonzero_mask, ["filename", "label"]].reset_index(drop=True)


def file_number(name):

    return int(re.search(r"(\d+)", str(name)).group(1))


groups = nonzero_df["filename"].map(file_number).values // 3


group_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)


group_folds = list(group_cv.split(X, strata, groups))


best_config = next(c for c in configs if c["name"] == BEST_WAV_MODEL)


group_oof = np.zeros(len(y))


for fold, (tr, va) in enumerate(group_folds):

    model = make_model(best_config)

    model.fit(X[tr], y[tr])

    group_oof[va] = model.predict(X[va])


group_rmse, group_r, group_proxy = score(y, group_oof)


random_best_oof = wavlm_predictions[BEST_WAV_MODEL]


random_rmse, random_r, random_proxy = score(y, random_best_oof)


print("\n" + "=" * 80)
print("WAVLM GROUP-OF-3 STRESS TEST")
print("=" * 80)


print("Model:", BEST_WAV_MODEL)


print("\nRandom stratified CV:")

print("RMSE   :", round(random_rmse, 6))

print("Pearson:", round(random_r, 6))

print("Proxy  :", round(random_proxy, 6))


print("\nGroup-of-3 CV:")

print("RMSE   :", round(group_rmse, 6))

print("Pearson:", round(group_r, 6))

print("Proxy  :", round(group_proxy, 6))


# 13. SAVE OOF RESULTS

np.save(WORK / "wavlm_best_oof.npy", random_best_oof)


pd.DataFrame(
    {
        "filename": nonzero_df["filename"],
        "label": y,
        "wavlm_oof": random_best_oof,
        "nested_wavlm_blend": nested_pred,
    }
).to_csv(WORK / "wavlm_oof_results.csv", index=False)


blend_table.to_csv(WORK / "wavlm_blend_search.csv", index=False)


print("\nSaved:")
print(WORK / "wavlm_best_oof.npy")

print(WORK / "wavlm_oof_results.csv")

print(WORK / "wavlm_blend_search.csv")

Best WavLM SVR OOF: RMSE=0.57904, Pearson=0.82152
Grouped stress test: RMSE=0.60454, Pearson=0.80371


In [15]:
# STEP 13C — EXTEND WAVLM WEIGHT SEARCH
#
# Previous search stopped at 0.40 and optimum hit boundary.
# Search 0.30 -> 0.85.

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

BEST_WAV_MODEL = "svr_C5.0_e0.1"

wav_oof = wavlm_predictions[BEST_WAV_MODEL]


def score_ext(y_true, pred):
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r = pearsonr(y_true, pred)[0]
    proxy = (rmse + (1 - r)) / 2
    return rmse, r, proxy


def crossfit_cal(raw_pred):
    out = np.zeros(len(y))

    for tr, va in folds:
        cal = LinearRegression()

        cal.fit(raw_pred[tr].reshape(-1, 1), y[tr])

        out[va] = cal.predict(raw_pred[va].reshape(-1, 1))

    return np.clip(out, 1.0, 5.0)


# 1. FIXED-WEIGHT SWEEP

weights = np.arange(0.30, 0.851, 0.025)

rows = []


for w in weights:

    raw = w * wav_oof + (1 - w) * current_raw

    pred = crossfit_cal(raw)

    rmse, r, proxy = score_ext(y, pred)

    rows.append(
        {
            "wavlm_weight": w,
            "text_weight": 1 - w,
            "rmse": rmse,
            "pearson": r,
            "proxy": proxy,
        }
    )


extended_table = (
    pd.DataFrame(rows).sort_values(["proxy", "rmse"]).reset_index(drop=True)
)


print("=" * 80)
print("EXTENDED WAVLM WEIGHT SWEEP")
print("=" * 80)

print(extended_table.head(20).round(6).to_string(index=False))


# 2. NESTED SELECTION

nested_ext = np.zeros(len(y))
nested_rows = []


for outer_fold, (_, va) in enumerate(folds):

    meta_train = np.setdiff1d(np.arange(len(y)), va)

    best = None

    for w in weights:

        raw = w * wav_oof + (1 - w) * current_raw

        cal = LinearRegression()

        cal.fit(raw[meta_train].reshape(-1, 1), y[meta_train])

        p = cal.predict(raw[meta_train].reshape(-1, 1))

        p = np.clip(p, 1.0, 5.0)

        rmse, r, proxy = score_ext(y[meta_train], p)

        candidate = (proxy, rmse, w)

        if best is None or candidate[:2] < best[:2]:
            best = candidate

    _, _, chosen_w = best

    raw = chosen_w * wav_oof + (1 - chosen_w) * current_raw

    cal = LinearRegression()

    cal.fit(raw[meta_train].reshape(-1, 1), y[meta_train])

    nested_ext[va] = cal.predict(raw[va].reshape(-1, 1))

    nested_ext[va] = np.clip(nested_ext[va], 1.0, 5.0)

    rmse, r, proxy = score_ext(y[va], nested_ext[va])

    nested_rows.append(
        {
            "fold": outer_fold,
            "wavlm_weight": chosen_w,
            "text_weight": 1 - chosen_w,
            "rmse": rmse,
            "pearson": r,
            "proxy": proxy,
        }
    )


nested_rmse, nested_r, nested_proxy = score_ext(y, nested_ext)


print("\n" + "=" * 80)
print("NESTED EXTENDED WEIGHT CHECK")
print("=" * 80)

print(pd.DataFrame(nested_rows).round(6).to_string(index=False))

print("\nNested RMSE   :", round(nested_rmse, 6))

print("Nested Pearson:", round(nested_r, 6))

print("Nested proxy  :", round(nested_proxy, 6))


# 3. BEST GLOBAL WEIGHT + CALIBRATION

best_w = float(extended_table.iloc[0]["wavlm_weight"])

best_raw = best_w * wav_oof + (1 - best_w) * current_raw


final_cal = LinearRegression()

final_cal.fit(best_raw.reshape(-1, 1), y)


BEST_WAV_WEIGHT = best_w
BEST_TEXT_WEIGHT = 1 - best_w

BEST_WAV_SLOPE = float(final_cal.coef_[0])

BEST_WAV_INTERCEPT = float(final_cal.intercept_)


print("\n" + "=" * 80)
print("BEST EXTENDED CONFIG")
print("=" * 80)

print("WavLM weight:", BEST_WAV_WEIGHT)

print("Text weight :", BEST_TEXT_WEIGHT)

print("Slope       :", round(BEST_WAV_SLOPE, 8))

print("Intercept   :", round(BEST_WAV_INTERCEPT, 8))

Final nested blend OOF: RMSE=0.532446, Pearson=0.851207
Selected weights: WavLM=0.575, text=0.425


## 12. Final fit and submission

The final text signal is `0.30 DeBERTa + 0.42 MPNet/numeric + 0.28 TF‑IDF`. It is blended with WavLM, transformed by the OOF-derived affine calibration, clipped to the valid rubric interval, and written in validated `test.csv` order.


In [16]:
# STEP 13D — FINAL WAVLM + TEXT SUBMISSION
#
# Final recipe:
#
# WavLM SVR:
#   StandardScaler
#   RBF SVR(C=5.0, epsilon=0.1)
#
# Text raw:
#   0.30 DeBERTa
# + 0.42 MPNet+numeric
# + 0.28 TF-IDF
#
# Final raw:
#   0.575 WavLM
# + 0.425 Text
#
# Then:
#   y = 1.18348009 * raw - 0.64239718
#   clip [1, 5]

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

WORK = Path("/kaggle/working")

DATA_DIR = Path(
    "/kaggle/input/competitions/" "shl-hiring-assessment-2026/" "Dataset_Final"
)


# 1. LOAD DATA

train_csv = pd.read_csv(DATA_DIR / "train.csv")

test_csv = pd.read_csv(DATA_DIR / "test.csv")


nonzero_mask = train_csv["label"].values > 0

y_final = train_csv.loc[nonzero_mask, "label"].values.astype(float)


wavlm_train_all = np.load(WORK / "wavlm_cache/" "train_wavlm_base_plus.npy")

wavlm_test = np.load(WORK / "wavlm_cache/" "test_wavlm_base_plus.npy")


X_wav_train = wavlm_train_all[nonzero_mask].astype(np.float32)

X_wav_test = wavlm_test.astype(np.float32)


print("=" * 80)
print("FINAL WAVLM DATA")
print("=" * 80)

print("Train:", X_wav_train.shape)

print("Test :", X_wav_test.shape)

print("Labels:", y_final.shape)


# 2. TRAIN FINAL WAVLM REGRESSOR

wavlm_final_model = make_pipeline(
    StandardScaler(), SVR(kernel="rbf", C=5.0, epsilon=0.1, gamma="scale")
)


print("\nTraining final WavLM SVR...")


wavlm_final_model.fit(X_wav_train, y_final)


wavlm_test_pred = wavlm_final_model.predict(X_wav_test)


print("\n" + "=" * 80)
print("WAVLM TEST PREDICTIONS")
print("=" * 80)

print(pd.Series(wavlm_test_pred).describe().round(6))


# 3. LOAD TEST COMPONENTS
#
# final_test_diagnostics.csv contains:
#   tfidf
#   mpnet_numeric
#   deberta

diag = pd.read_csv(WORK / "final_test_diagnostics.csv")


assert diag["filename"].tolist() == test_csv["filename"].tolist()


# 4. RECONSTRUCT SELECTED TEXT ENSEMBLE
#
# DeBERTa 0.30
# MPNet   0.42
# TF-IDF  0.28
#
# IMPORTANT:
# use RAW component predictions.
# Do not use already-calibrated "final".

text_raw_test = (
    0.30 * diag["deberta"].values
    + 0.42 * diag["mpnet_numeric"].values
    + 0.28 * diag["tfidf"].values
)


print("\n" + "=" * 80)
print("TEXT RAW TEST")
print("=" * 80)

print(pd.Series(text_raw_test).describe().round(6))


# 5. WAVLM + TEXT BLEND

WAVLM_WEIGHT = 0.575
TEXT_WEIGHT = 0.425


raw_final = WAVLM_WEIGHT * wavlm_test_pred + TEXT_WEIGHT * text_raw_test


print("\n" + "=" * 80)
print("RAW WAVLM + TEXT")
print("=" * 80)

print(pd.Series(raw_final).describe().round(6))


# 6. OOF-DERIVED CALIBRATION

CAL_SLOPE = 1.18348009
CAL_INTERCEPT = -0.64239718


calibrated = CAL_SLOPE * raw_final + CAL_INTERCEPT


final_pred = np.clip(calibrated, 1.0, 5.0)


# 7. SUBMISSION

submission = pd.DataFrame({"filename": test_csv["filename"], "label": final_pred})


OUTPUT_PATH = WORK / "submission.csv"


submission.to_csv(OUTPUT_PATH, index=False)


# 8. DIAGNOSTICS

full_diag = pd.DataFrame(
    {
        "filename": test_csv["filename"],
        "wavlm": wavlm_test_pred,
        "text_raw": text_raw_test,
        "raw_blend": raw_final,
        "calibrated": calibrated,
        "final": final_pred,
    }
)


full_diag.to_csv(WORK / "wavlm_final_test_diagnostics.csv", index=False)


print("\n" + "=" * 80)
print("FINAL SUBMISSION SUMMARY")
print("=" * 80)


print(submission["label"].describe().round(6))


print("\nClipped to 1:", int((calibrated < 1).sum()))


print("Clipped to 5:", int((calibrated > 5).sum()))


# 9. MODEL AGREEMENT

print("\n" + "=" * 80)
print("TEST MODEL AGREEMENT")
print("=" * 80)


print(
    "Correlation WavLM vs text:",
    round(float(np.corrcoef(wavlm_test_pred, text_raw_test)[0, 1]), 6),
)


full_diag["abs_disagreement"] = np.abs(full_diag["wavlm"] - full_diag["text_raw"])


print("\nLargest disagreements:")

print(
    full_diag.sort_values("abs_disagreement", ascending=False)
    .head(20)
    .round(4)
    .to_string(index=False)
)


# 10. EXTREMES

print("\n" + "=" * 80)
print("LOWEST FINAL SCORES")
print("=" * 80)

print(full_diag.sort_values("final").head(15).round(4).to_string(index=False))


print("\n" + "=" * 80)
print("HIGHEST FINAL SCORES")
print("=" * 80)

print(
    full_diag.sort_values("final", ascending=False)
    .head(15)
    .round(4)
    .to_string(index=False)
)


print("\nSaved:")
print(OUTPUT_PATH)

print(WORK / "wavlm_final_test_diagnostics.csv")

Final submission: 216 unique rows, no missing values, predictions clipped to [1, 5].
Saved: /kaggle/working/submission.csv


## 13. Required metric and diagnostics

The competition explicitly asks for training-data RMSE. I report **out-of-fold RMSE**, not an in-sample training score, because it is the more honest estimate of generalization. The following assertions also fail loudly if the final file has the wrong order, row count, duplicates, missing values, or out-of-range predictions.


In [17]:
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

final_oof = nested_ext
final_oof_rmse = mean_squared_error(y, final_oof) ** 0.5
final_oof_pearson = pearsonr(y, final_oof).statistic

assert len(submission) == len(test_csv) == 216
assert submission["filename"].tolist() == test_csv["filename"].tolist()
assert submission["filename"].is_unique
assert submission["label"].between(1.0, 5.0).all()
assert submission["label"].notna().all()

print("REQUIRED TRAINING-DATA OOF METRICS")
print(f"RMSE:    {final_oof_rmse:.6f}")
print(f"Pearson: {final_oof_pearson:.6f}")
print(f"Submission: {WORK_DIR / 'submission.csv'}")

residuals = y - final_oof
fold_ids = np.empty(len(y), dtype=int)
for fold_id, (_, validation_indices) in enumerate(folds):
    fold_ids[validation_indices] = fold_id

fold_rmse = [
    mean_squared_error(y[fold_ids == fold_id], final_oof[fold_ids == fold_id]) ** 0.5
    for fold_id in range(N_SPLITS)
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].scatter(y, final_oof, alpha=0.45, s=20)
axes[0, 0].plot([1, 5], [1, 5], "--", color="black", linewidth=1)
axes[0, 0].set(
    title="OOF: observed vs predicted", xlabel="Observed", ylabel="Predicted"
)

axes[0, 1].scatter(final_oof, residuals, alpha=0.45, s=20)
axes[0, 1].axhline(0, linestyle="--", color="black", linewidth=1)
axes[0, 1].set(title="OOF residuals", xlabel="Predicted", ylabel="Observed − predicted")

axes[1, 0].bar(range(N_SPLITS), fold_rmse, color="#70AD47")
axes[1, 0].axhline(final_oof_rmse, linestyle="--", color="black", label="overall")
axes[1, 0].set(title="RMSE by fold", xlabel="Fold", ylabel="RMSE")
axes[1, 0].legend()

axes[1, 1].hist(submission["label"], bins=16, color="#ED7D31", alpha=0.85)
axes[1, 1].set(title="Final test predictions", xlabel="Predicted score", ylabel="Count")

plt.tight_layout()
plt.show()

REQUIRED TRAINING-DATA OOF METRICS
RMSE:    0.532446
Pearson: 0.851207
Submission: /kaggle/working/submission.csv


## 14. What worked, what did not, and remaining risk

**What worked.** Transcript-based models made the largest early improvement; MPNet and DeBERTa helped because their errors were not identical to sparse TF‑IDF. WavLM produced the largest final gain by restoring information lost during transcription. Nested selection showed the audio/text gain was stable across folds.

**What did not.** Handcrafted acoustics were confounded by the anomalous zero-score batch, CoLA features did not transfer well from written text, and a dedicated zero-regime detector found almost no similar test examples. Those branches were investigated but excluded from the final prediction path.

**Limitations.** The dataset is small, ASR can normalize or hallucinate grammatical forms, and random stratified CV can still be optimistic if speakers repeat. The grouped WavLM stress test remained useful (RMSE 0.60454) but was weaker than random CV, so the headline 0.53245 should be interpreted with that caution. The final Kaggle score is intentionally not claimed here because it was not exported with the notebook artifacts.
